# CSI-WiFormer — ablation trên Kaggle (bản đầy đủ code)

Notebook này **chứa sẵn toàn bộ code** (các ô `%%writefile`), không cần GitHub. Internet chỉ cần để cài `thop` (đo FLOPs, tuỳ chọn).

1. **Add Input** → dataset SSHAR (thư mục chứa `room_*/esp/rx_00/...`), ví dụ `phanvanhai/csi-room-02-0707`.
2. **Settings** → Accelerator **GPU T4 x2** (hoặc P100).
3. Sửa ô *Cấu hình* → **Save Version → Save & Run All (Commit)** để chạy nền.
4. Kết quả trong tab **Output**: `results/<RUN_TAG>/...` và `results_<RUN_TAG>.zip`.

**Chia dữ liệu (mặc định mới):**
- `repeat`: train:val:test = 7:1:1 ngẫu nhiên theo lớp (seed cố định 42) — mọi biến thể dùng chung tập test.
- `position` / `user`: test = vị trí / người bị giữ lại; phần còn lại chia train:val = 7:1.
- Epoch tốt nhất chọn theo **val**, test chỉ đánh giá 1 lần ở cuối. **Dừng sớm** khi val acc không tăng sau 10 epoch (`--patience=10`, đặt `--patience=0` trong `EXTRA` để tắt). Optimizer mặc định: **Adam, lr = 1e-3** (WiFormer có warmup 5 epoch + cosine).

**Giới hạn 12 giờ/phiên:** 1 run WiFormer 100 epoch trên SSHAR-ESP mất ~2–3 giờ (T4), nên **1 phiên chỉ chạy được ~3–4 biến thể** (dừng sớm thường rút ngắn đáng kể, xem cột `epochs_trained` ở bảng tổng hợp).
Chia `VARIANTS` ra nhiều phiên; muốn gộp/tiếp tục thì thêm Output của phiên trước làm Input và đặt `PREV_RESULTS` (biến thể đã xong sẽ được bỏ qua).

## Cấu hình

In [ ]:
CODE_DIR     = '/kaggle/working/cmr_har_code'   # KHÔNG đổi (các ô %%writefile ghi vào đây)
DATA_ROOT    = ''         # để trống -> tự tìm trong /kaggle/input (thư mục chứa room_*)
OUTPUT       = '/kaggle/working'

DATASET    = 'sshar_esp'  # sshar_esp | sshar_asus | xrf55 | ut_har
SPLIT_MODE = 'repeat'     # repeat | position | user
CASES      = [0]          # position: 0..3, user: 0..4; repeat: bỏ qua (chỉ chạy 1 lần)

# Biến thể ablation (python ablation.py --list):
#   core  : full wo_fft wo_conv wo_block1 wo_block2 wo_block3
#   design: k2_per_rx k2_pe_none k2_pe_rx_sub k3_learned k3_nopos k2_l1 k3_l2 no_norm all_ant win32 win128
#   scale : k1_flat4 k1_deep k2_l3 k3_l6 d96 d128 d128_k1deep   (tăng độ sâu/rộng)
VARIANTS = ['full', 'wo_block2', 'wo_block3']     # ~3 biến thể / phiên 12h
EXTRA    = '--epochs=100'                          # thêm tham số cho main.py, vd '--lr=5e-4'
# EXTRA  = '--epochs=2'                            # chạy thử nhanh

RUN_TAG      = 'ablation_v2'   # thư mục kết quả: results/<RUN_TAG>/<split>_c<case>/
PREV_RESULTS = ''              # vd '/kaggle/input/wiformer-ablation-v2/results/ablation_v2' để chạy tiếp

## Môi trường + dữ liệu

In [ ]:
!nvidia-smi
!nproc; free -g; df -h /kaggle/working | cat
!find /kaggle/input -maxdepth 4 -type d | head -40

In [ ]:
import os
from pathlib import Path

def find_sshar(root='/kaggle/input', max_depth=6):
    """Thư mục chứa các thư mục room_* (có <device>/rx_00 bên trong)."""
    for depth in range(1, max_depth + 1):
        for p in sorted(Path(root).glob('/'.join(['*'] * depth))):
            if p.is_dir() and p.name.startswith('room') and any(p.glob('*/rx_00')):
                return str(p.parent)
    return None

if not DATA_ROOT:
    DATA_ROOT = find_sshar() if DATASET.startswith('sshar') else '/kaggle/input'
    assert DATA_ROOT, 'Không tìm thấy room_*/<device>/rx_00 trong /kaggle/input — kiểm tra Add Input'
print('DATA_ROOT =', DATA_ROOT)
!ls {DATA_ROOT} | head -20

## Ghi code ra đĩa

Các ô dưới đây ghi lại toàn bộ code của repo (nhánh `ablation`). Sửa code thì sửa trực tiếp trong ô tương ứng.

In [ ]:
import os
os.makedirs('/kaggle/working/cmr_har_code/models', exist_ok=True)

In [ ]:
%%writefile /kaggle/working/cmr_har_code/datasets.py
"""
datasets.py

Unified dataset loader for WiFi HAR Benchmark

Supported datasets
------------------
1. UT_HAR
2. SSHAR_ESP
3. SSHAR_asus
4. XRF55
"""

from pathlib import Path
import os
import glob
import re

import numpy as np

import torch
import torch.nn.functional as F

from torch.utils.data import Dataset
from torch.utils.data import DataLoader


# Default split mode when none is passed: "repeat" (by repetition index),
# "user" (cross-user, train users given by USER_CASES[...][case_split]) or
# "position" (SSHAR only, cross-position: test = positions with
# pos % N_POSITION_CASES == case_split, i.e. 2 of the 8 positions per case).
SPLIT_MODES = ("repeat", "user", "position")
N_POSITION_CASES = 4
DEFAULT_SPLIT_MODE = "repeat"


def _check_split_mode(split_mode):
    split_mode = (split_mode or DEFAULT_SPLIT_MODE).lower()
    if split_mode not in SPLIT_MODES:
        raise ValueError(f"split_mode must be one of {SPLIT_MODES}, got '{split_mode}'")
    return split_mode


# train : val : test ratio.
#   repeat           -> all samples split 7:1:1, stratified by class.
#   user / position  -> test = held-out users/positions (unchanged);
#                       the rest is split train:val = 7:1, stratified by class.
# The split uses its own fixed seed, so it is identical across runs/models.
SPLIT_RATIO = (7, 1, 1)
SPLIT_SEED = 42


def _assign_splits(labels, held_out, carve_test, ratio=SPLIT_RATIO, seed=SPLIT_SEED):
    """Return an array of 'train' / 'val' / 'test', one per sample.

    labels   : class index per sample (samples must be in a deterministic order)
    held_out : True if the sample is already test (user / position split)
    carve_test : True for 'repeat' (test is a random 1/9); False for user /
                 position (test = held_out only, rest split train:val)
    """
    labels = np.asarray(labels)
    held_out = np.asarray(held_out, dtype=bool)
    out = np.full(len(labels), "train", dtype=object)
    out[held_out] = "test"

    n_train, n_val, n_test = ratio
    if not carve_test:
        n_test = 0  # test already fixed, only carve val out of the rest
    total = n_train + n_val + n_test

    rng = np.random.RandomState(seed)
    for c in np.unique(labels):
        idx = np.where((labels == c) & ~held_out)[0]
        rng.shuffle(idx)
        k_test = int(round(len(idx) * n_test / total))
        k_val = int(round(len(idx) * n_val / total))
        out[idx[:k_test]] = "test"
        out[idx[k_test:k_test + k_val]] = "val"
    return out

# ============================================================
# Default configuration
# ============================================================

DEFAULT_BATCH_SIZE = 32
DEFAULT_NUM_WORKERS = 0
DEFAULT_PIN_MEMORY = True

# Keep only the requested classes in the benchmark.
# Labels are remapped to contiguous indices 0..5 for PyTorch.
DEFAULT_SELECTED_LABELS = (1, 2, 3, 4, 5, 6, 7, 8)
DEFAULT_LABEL_MAP = {
    label: idx for idx, label in enumerate(DEFAULT_SELECTED_LABELS)
}

INTEL_30_SUBCARRIERS = [
    -28, -26, -24, -22, -20, -18, -16, -14,
    -12, -10, -8, -6, -4, -2,
    -1, 1, 3, 5, 7, 9, 11, 13,
    15, 17, 19, 21, 23, 25, 27, 28
]

# Map signed subcarrier -> index 0..55
INTEL_30_INDICES = [
    sc + 28 if sc < 0 else sc + 27
    for sc in INTEL_30_SUBCARRIERS
]

USER_CASES = {
    "xrf55": [
        list(range(1, 25)),
        [1, 2, 3, 5, 6, 7, 8, 9, 10, 11, 12, 13, 15, 16, 18, 19, 21, 23, 24, 25, 27, 28, 29, 30],
        [1, 2, 4, 5, 6, 7, 8, 9, 10, 13, 14, 16, 17, 18, 19, 20, 22, 23, 24, 25, 26, 27, 28, 29],
        [1, 2, 4, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 18, 19, 22, 23, 24, 25, 26, 27, 28, 29],
        [1, 2, 4, 5, 6, 7, 8, 9, 10, 11, 13, 14, 15, 16, 17, 18, 19, 21, 23, 24, 25, 26, 29, 30]
    ],
    "sshar": [
        [1, 2, 3, 4, 9, 10, 11, 12],
        [1, 3, 4, 9, 11, 12, 13, 14],
        [1, 2, 4, 9, 10, 12, 13, 14],
        [2, 3, 4, 10, 11, 12, 13, 14],
        [1, 2, 4, 9, 10, 11, 13, 14],
    ]
}
# ============================================================
# Utility
# ============================================================

def scan_folders(root: Path, prefix: str):
    """
    Scan folders with given prefix.

    Example
    -------
    scan_folders(data_root, "room")
        -> ["room_01","room_02"]

    scan_folders(room_path, "subject")
        -> ["subject_01",...]
    """

    if not root.exists():
        return []

    folders = []

    for item in sorted(root.iterdir()):

        if item.is_dir() and item.name.startswith(prefix):
            folders.append(item.name)

    return folders


# ============================================================
#                    UT_HAR
# ============================================================

class UTHARDataset(Dataset):
    """
    UT_HAR Dataset

    Folder structure

    UT_HAR
    ├── data
    │     train_data.csv
    │     val_data.csv
    │     test_data.csv
    │
    └── label
          train_label.csv
          val_label.csv
          test_label.csv

    Parameters
    ----------
    root_dir

    split
        train
        val
        test

    normalize

    norm_type
        minmax
        zscore

    mean,std
        only used for zscore
    """

    def __init__(
        self,
        root_dir,
        split="train",
        normalize=True,
        norm_type="minmax",
        mean=None,
        std=None,
    ):

        self.root_dir = Path(root_dir)
        self.split = split
        self.normalize = normalize
        self.norm_type = norm_type
        self.mean = mean
        self.std = std

        self.X = None
        self.y = None

        self.load_data()

    # --------------------------------------------------------

    def _find_ut_har_dir(self):
        """
        Locate the folder that contains data/X_{split}.csv and label/y_{split}.csv.
        Accepts root_dir pointing at the parent of UT_HAR, at UT_HAR itself,
        or anywhere above it (e.g. a Kaggle input dir with nested folders).
        """
        fname = f"X_{self.split}.csv"
        for cand in (self.root_dir / "UT_HAR", self.root_dir):
            if (cand / "data" / fname).exists():
                return cand

        if self.root_dir.exists():
            for hit in sorted(self.root_dir.rglob(fname)):
                if hit.parent.name == "data":
                    return hit.parent.parent

        raise FileNotFoundError(
            f"Could not find data/{fname} under '{self.root_dir}'. "
            f"Check --data_root (it should point at the UT_HAR folder or one of its parents)."
        )

    def load_data(self):
        base = self._find_ut_har_dir()
        data_file = base / "data" / f"X_{self.split}.csv"
        label_file = base / "label" / f"y_{self.split}.csv"

        if not label_file.exists():
            raise FileNotFoundError(label_file)

        with open(data_file, "rb") as f:
            x = np.load(f)

        # Raw data shape: (N, 250, 90) = (N, time, subcarrier)
        # Transpose to (N, 90, 250) = (N, subcarrier, time)
        # to match model input format (C, T)
        x = x.transpose(0, 2, 1)

        with open(label_file, "rb") as f:
            y = np.load(f)

        # one-hot -> class index
        if len(y.shape) > 1:
            y = np.argmax(y, axis=1)

        self.X = x.astype(np.float32)
        self.y = y.astype(np.int64)

    # --------------------------------------------------------
    def __len__(self):
        return len(self.X)
    
    # --------------------------------------------------------
    def __getitem__(self, idx):
        x = self.X[idx].copy()
        if self.normalize:
            if self.norm_type == "minmax":
                xmin = x.min()
                xmax = x.max()
                x = (
                    x - xmin
                ) / (
                    xmax - xmin + 1e-8
                )
            elif self.norm_type == "zscore":
                if self.mean is None or self.std is None:
                    raise ValueError(
                        "mean/std required for zscore"
                    )
                x = (
                    x - self.mean
                ) / (
                    self.std + 1e-8
                )
            else:
                raise ValueError(
                    f"Unknown norm_type {self.norm_type}"
                )

        x = torch.from_numpy(x).float()
        y = int(self.y[idx])

        return x, y


# ============================================================
# compute mean/std for UT_HAR
# ============================================================
def compute_ut_har_mean_std(root_dir):
    """
    Compute global mean/std
    using TRAIN split only.

    Only used when norm_type='zscore'
    """

    dataset = UTHARDataset(
        root_dir=root_dir,
        split="train",
        normalize=False,
    )

    total_sum = 0.0
    total_sq = 0.0
    total_count = 0

    n = len(dataset)
    print(f"[XRF55] computing mean/std over {n} train samples ...")
    for i, (x, _) in enumerate(dataset):
        if (i + 1) % 500 == 0 or i + 1 == n:
            print(f"[XRF55]   mean/std {i + 1}/{n}")
        x = x.numpy().astype(np.float64)
        total_sum += x.sum()
        total_sq += np.square(x).sum()
        total_count += x.size

    mean = total_sum / total_count

    std = np.sqrt(
        total_sq / total_count - mean ** 2
    )

    print(f"UT_HAR mean = {mean:.6f}")
    print(f"UT_HAR std  = {std:.6f}")

    return mean, std


# ============================================================
#               Base SSHAR Dataset
# ============================================================
class _SSHARDatasetBase(Dataset):
    """
    Internal SSHAR dataset.

    Do not use directly.

    Use

        SSHAR_ESP_Dataset

    or

        SSHAR_ASUS_Dataset
    """

    pattern = re.compile(
        r"act(\d+)_pos(\d+)_dir(\d+)_rep(\d+)"
    )
    # --------------------------------------------------------

    def __init__(
        self,
        root_dir,
        device,
        signal="amp",
        split="train",
        case_split = 0,
        split_mode=DEFAULT_SPLIT_MODE,
        shape_option="2d",
        normalize=True,
        norm_type="zscore",
        mean=None,
        std=None,
        target_time=None,
        rooms=None,
        subjects=None,
        rx_list=None,
        intel_shape=False,
        allowed_labels=None,
    ):

        self.root_dir = Path(root_dir)
        self.device = device
        self.signal = signal
        self.split = split
        self.case_split = case_split
        self.split_mode = _check_split_mode(split_mode)
        self.shape_option = shape_option
        self.normalize = normalize
        self.norm_type = norm_type
        self.mean = mean
        self.std = std
        self.target_time = target_time
        self.intel_shape = intel_shape
        self.allowed_labels = tuple(DEFAULT_SELECTED_LABELS) if allowed_labels is None else tuple(allowed_labels)
        self.label_map = {label: idx for idx, label in enumerate(self.allowed_labels)}
        # ----------------------------------------
        if rooms is None:
            rooms = scan_folders(
                self.root_dir,
                "room",
            )

        self.rooms = rooms

        # ----------------------------------------
        if rx_list is None:
            rx_list = [
                "rx_00",
                "rx_01",
                "rx_02",
            ]
        self.rx_list = rx_list

        # ----------------------------------------
        if subjects is None:
            subjects = []
            for room in self.rooms:
                base = (
                    self.root_dir /
                    room /
                    self.device /
                    self.rx_list[0]
                )

                subjects.extend(
                    scan_folders(
                        base,
                        "subject",
                    )
                )

            subjects = sorted(
                list(
                    set(subjects)
                )
            )

        self.subjects = subjects

        # ----------------------------------------

        self.samples = []
        self.build_index()

    # --------------------------------------------------------
    def build_index(self):

        """
        Build sample index.

        Only scan RX00.

        Store all RX file paths.

        samples[i] =

        {
            "label":0,
            "files":[
                rx00,
                rx01,
                rx02
            ]
        }
        """
        self.samples.clear()
        for room in self.rooms:
            for subject in self.subjects:
                folder = (
                    self.root_dir /
                    room /
                    self.device /
                    self.rx_list[0] /
                    subject
                )

                if not folder.exists():
                    continue

                for file in sorted(
                    folder.iterdir()
                ):
                    if not file.name.startswith(
                        self.signal
                    ):
                        continue

                    m = self.pattern.search(
                        file.name
                    )
                    if m is None:
                        continue

                    act = int(m.group(1))
                    pos = int(m.group(2))
                    direction = int(m.group(3))
                    rep = int(m.group(4))

                    if act not in self.label_map:
                        continue

                    
                    if self.split_mode == "repeat":
                        # Chia ngau nhien 7:1:1 theo lop (xem _assign_splits)
                        held_out = False
                    elif self.split_mode == "position":
                        # Chia theo vi tri: giu lai 2/8 vi tri lam test
                        held_out = pos % N_POSITION_CASES == self.case_split
                    else:
                        # Chia theo nguoi
                        user_num = int(subject.split("_")[1])
                        held_out = user_num not in USER_CASES["sshar"][self.case_split]

                    rx_files = []

                    for rx in self.rx_list:
                        rx_files.append(
                            self.root_dir
                            / room
                            / self.device
                            / rx
                            / subject
                            / file.name
                        )

                    self.samples.append(
                        {
                            "label": self.label_map[act],
                            "room": room,
                            "subject": subject,
                            "pos": pos,
                            "dir": direction,
                            "rep": rep,
                            "files": rx_files,
                            "held_out": held_out,
                        }
                    )

        splits = _assign_splits(
            [s["label"] for s in self.samples],
            [s["held_out"] for s in self.samples],
            carve_test=self.split_mode == "repeat",
        )
        self.samples = [s for s, sp in zip(self.samples, splits) if sp == self.split]

    # --------------------------------------------------------
    def __len__(self):
        return len(self.samples)

    # --------------------------------------------------------
    def __getitem__(self, idx):
        sample = self.samples[idx]
        rx_data = []

        for file in sample["files"]:
            x = np.load(file).astype(np.float32)
            rx_data.append(x)

        # --------------------------------------------------
        # Stack RX
        #
        # ESP:
        #   (3, 1, 56, T)
        #
        # ASUS:
        #   (3, 4, 56, T)
        # --------------------------------------------------
        x = np.stack(rx_data)

        # ============================================================
        # Intel shape
        # ============================================================
        if self.intel_shape:
            # Select antennas
            if self.device == "esp":
                # ESP32: 1 antenna
                antenna_indices = [0]
            elif self.device == "asus":
                # ASUS: keep antenna 0, 1, 3
                antenna_indices = [0,1,2,3]
            else:
                raise ValueError(
                    f"Unknown device: {self.device}"
                )
            x = x[:, antenna_indices, :, :]
            # Select 30 Intel-like subcarriers
            # x = x[:, :, INTEL_30_INDICES, :]

        # --------------------------------------------------
        # Shape
        # --------------------------------------------------
        if self.shape_option == "2d":
            # (rx, antenna, sub, time)
            # -> (rx * antenna * sub, time)
            x = x.reshape(
                -1,
                x.shape[-1]
            )
        elif self.shape_option == "3d":
            # (rx, antenna, sub, time)
            # -> (rx * antenna, sub, time)
            rx, antenna, sub, time_len = x.shape
            x = x.reshape(
                rx * antenna,
                sub,
                time_len
            )
        else:
            raise ValueError(
                f"Unknown shape_option: {self.shape_option}"
            )

        # --------------------------------------------------
        # Normalization
        # --------------------------------------------------
        if self.normalize:
            if self.norm_type == "zscore":
                if self.mean is None or self.std is None:
                    raise ValueError(
                        "mean/std required for zscore"
                    )
                x = (
                    x - self.mean
                ) / (
                    self.std + 1e-8
                )
            elif self.norm_type == "minmax":
                xmin = x.min()
                xmax = x.max()
                x = (
                    x - xmin
                ) / (
                    xmax - xmin + 1e-8
                )
            else:
                raise ValueError(
                    f"Unknown norm_type {self.norm_type}"
                )

        x = torch.from_numpy(x).float()

        if self.target_time is not None and x.ndim in (2, 3) and x.shape[-1] != self.target_time:
            x = F.adaptive_max_pool1d(x, output_size=self.target_time)

        y = sample["label"]
        return x, y


# ============================================================
# SSHAR ESP
# ============================================================

class SSHAR_ESP_Dataset(_SSHARDatasetBase):
    def __init__(
        self,
        root_dir,
        signal="amp",
        split="train",
        case_split = 0,
        split_mode=DEFAULT_SPLIT_MODE,
        shape_option="2d",
        normalize=True,
        norm_type="zscore",
        mean=None,
        std=None,
        target_time=None,
        rooms=None,
        subjects=None,
        rx_list=None,
        intel_shape=False,
        allowed_labels=None,
    ):
        super().__init__(
            root_dir=root_dir,
            device="esp",
            signal=signal,
            split=split,
            case_split = case_split,
            split_mode=split_mode,
            shape_option=shape_option,
            normalize=normalize,
            norm_type=norm_type,
            mean=mean,
            std=std,
            target_time=target_time,
            rooms=rooms,
            subjects=subjects,
            rx_list=rx_list,
            intel_shape=intel_shape,
            allowed_labels=allowed_labels,
        )

# ============================================================
# SSHAR asus
# ============================================================
class SSHAR_ASUS_Dataset(_SSHARDatasetBase):
    def __init__(
        self,
        root_dir,
        signal="amp",
        split="train",
        case_split = 0,
        split_mode=DEFAULT_SPLIT_MODE,
        shape_option="2d",
        normalize=True,
        norm_type="zscore",
        mean=None,
        std=None,
        target_time=None,
        rooms=None,
        subjects=None,
        rx_list=None,
        intel_shape=False,
        allowed_labels=None,
    ):
        super().__init__(
            root_dir=root_dir,
            device="asus",
            signal=signal,
            split=split,
            case_split = case_split,
            split_mode=split_mode,
            shape_option=shape_option,
            normalize=normalize,
            norm_type=norm_type,
            mean=mean,
            std=std,
            target_time=target_time,
            rooms=rooms,
            subjects=subjects,
            rx_list=rx_list,
            intel_shape=intel_shape,
            allowed_labels=allowed_labels,
        )

# ============================================================
# Compute SSHAR mean/std
# ============================================================
def compute_sshar_mean_std(dataset):
    """
    Parameters
    ----------
    dataset

        SSHAR_ESP_Dataset
        or
        SSHAR_ASUS_Dataset

        normalize=False
    """

    total_sum = 0.0
    total_sq = 0.0
    total_count = 0

    n = len(dataset)
    print(f"[XRF55] computing mean/std over {n} train samples ...")
    for i, (x, _) in enumerate(dataset):
        if (i + 1) % 500 == 0 or i + 1 == n:
            print(f"[XRF55]   mean/std {i + 1}/{n}")
        x = x.numpy().astype(np.float64)
        total_sum += x.sum()
        total_sq += np.square(x).sum()
        total_count += x.size

    mean = total_sum / total_count
    std = np.sqrt(
        total_sq / total_count - mean ** 2
    )

    print(f"SSHAR mean = {mean:.6f}")
    print(f"SSHAR std  = {std:.6f}")

    return mean, std


# ============================================================
# XRF55 Dataset (Subset)
# ============================================================
_XRF55_FILE_CACHE = {}


def find_xrf55_wifi_files(root_dir):
    """
    Locate XRF55 WiFi .npy files.

    1. *.npy directly inside root_dir  (root_dir is the WiFi folder)
    2. otherwise *.npy inside sub-folders named 'WiFi' (case-insensitive),
       e.g. root/train_data/WiFi, root/test_data/WiFi.
       Other modalities (mmWave, RFID, ...) are never included.
    """
    root_dir = Path(root_dir)
    key = str(root_dir.resolve())
    if key in _XRF55_FILE_CACHE:
        return _XRF55_FILE_CACHE[key]

    files = sorted(root_dir.glob("*.npy"))
    if files:
        print(f"[XRF55] {len(files)} .npy files in {root_dir}")
    else:
        wifi_dirs = sorted(
            d for d in root_dir.rglob("*")
            if d.is_dir() and d.name.lower() == "wifi"
        )
        for d in wifi_dirs:
            found = sorted(d.glob("*.npy"))
            print(f"[XRF55] {len(found)} .npy files in {d}")
            files.extend(found)
        if not wifi_dirs:
            print(f"[XRF55] no .npy files in {root_dir} and no 'WiFi' sub-folder found")

    _XRF55_FILE_CACHE[key] = files
    return files


class XRF55Dataset(Dataset):
    """
    XRF55 Dataset (Subset for 8 actions and all users)

    Format requirements:
    - Files are in .npy format.
    - Filenames follow the pattern a_b_c.npy
      a = User ID (2 digits)
      b = Action ID (2 digits) -> will be zero-indexed for PyTorch
      c = Repetition number

    Split rule is determined by a maximum repetition number for training.
    """
    def __init__(
        self,
        root_dir,
        split="train",
        case_split=0,
        split_mode=DEFAULT_SPLIT_MODE,
        shape_option="2d",
        num_sub=30,  # XRF55 typically uses Intel 5300 with 30 subcarriers per antenna
        normalize=True,
        norm_type="minmax",
        mean=None,
        std=None,
        target_time=None,
        allowed_labels=None,
    ):
        self.root_dir = Path(root_dir)
        self.split = split
        self.case_split = case_split
        self.split_mode = _check_split_mode(split_mode)
        self.shape_option = shape_option
        self.num_sub = num_sub
        self.normalize = normalize
        self.norm_type = norm_type
        self.mean = mean
        self.std = std
        self.target_time = target_time
        self.allowed_labels = tuple(DEFAULT_SELECTED_LABELS) if allowed_labels is None else tuple(allowed_labels)
        self.label_map = {label: idx for idx, label in enumerate(self.allowed_labels)}

        self.samples = []
        self.build_index()

    def build_index(self):
        if not self.root_dir.exists():
            raise FileNotFoundError(f"Directory not found: {self.root_dir}")

        for file_path in find_xrf55_wifi_files(self.root_dir):
            name_parts = file_path.stem.split('_')
            
            # Ensure it matches a_b_c format
            if len(name_parts) == 3:
                user_id, action_id, rep_id = name_parts
                rep_num = int(rep_id)
                
                action_id_int = int(action_id)
                if action_id_int not in self.label_map:
                    continue

                if self.split_mode == "position":
                    raise ValueError("split_mode='position' is only available for SSHAR")
                if self.split_mode == "repeat":
                    # Chia ngau nhien 7:1:1 theo lop (xem _assign_splits)
                    held_out = False
                else:
                    # Chia theo nguoi
                    user_num = int(user_id)
                    held_out = user_num not in USER_CASES["xrf55"][self.case_split]

                # Remap selected labels to contiguous indices 0..5.
                label = self.label_map[action_id_int]

                self.samples.append({
                    "path": str(file_path),
                    "label": label,
                    "rep": rep_num,
                    "held_out": held_out,
                })

        splits = _assign_splits(
            [s["label"] for s in self.samples],
            [s["held_out"] for s in self.samples],
            carve_test=self.split_mode == "repeat",
        )
        self.samples = [s for s, sp in zip(self.samples, splits) if sp == self.split]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        x = np.load(sample["path"]).astype(np.float32)
        y = sample["label"]
        
        # Reshape logically based on shape_option
        if self.shape_option == "3d":
            # Assuming raw input is 2D: (C, Time) where C = dev_anten * num_sub
            if len(x.shape) == 2:
                c, t = x.shape
                assert c % self.num_sub == 0
                dev_anten = c // self.num_sub
                x = x.reshape(dev_anten, self.num_sub, t)
        elif self.shape_option == "2d":
            # If input is already 3D: (dev_anten, sub, Time), flatten it back to 2D
            if len(x.shape) == 3:
                d, s, t = x.shape
                x = x.reshape(d * s, t)

        if self.normalize:
            if self.norm_type == "minmax":
                xmin = x.min()
                xmax = x.max()
                x = (x - xmin) / (xmax - xmin + 1e-8)
            elif self.norm_type == "zscore":
                if self.mean is None or self.std is None:
                    raise ValueError("mean/std required for zscore")
                x = (x - self.mean) / (self.std + 1e-8)
            else:
                raise ValueError(f"Unknown norm_type {self.norm_type}")

        x = torch.from_numpy(x).float()

        if self.target_time is not None and x.ndim in (2, 3) and x.shape[-1] != self.target_time:
            x = F.adaptive_max_pool1d(x, output_size=self.target_time)

        return x, y


# ============================================================
# Compute XRF55 mean/std
# ============================================================
def compute_xrf55_mean_std(root_dir, case_split, shape_option="2d", num_sub=30, target_time=None,
                           split_mode=DEFAULT_SPLIT_MODE):
    """
    Compute global mean/std for XRF55 using TRAIN split only.
    """
    dataset = XRF55Dataset(
        root_dir=root_dir,
        split="train",
        case_split=case_split,
        split_mode=split_mode,
        shape_option=shape_option,
        num_sub=num_sub,
        normalize=False,
        target_time=target_time,
    )

    total_sum = 0.0
    total_sq = 0.0
    total_count = 0

    n = len(dataset)
    print(f"[XRF55] computing mean/std over {n} train samples ...")
    for i, (x, _) in enumerate(dataset):
        if (i + 1) % 500 == 0 or i + 1 == n:
            print(f"[XRF55]   mean/std {i + 1}/{n}")
        x = x.numpy().astype(np.float64)
        total_sum += x.sum()
        total_sq += np.square(x).sum()
        total_count += x.size

    if total_count == 0:
        return 0.0, 1.0

    mean = total_sum / total_count
    std = np.sqrt(total_sq / total_count - mean ** 2)

    print(f"XRF55 mean = {mean:.6f}")
    print(f"XRF55 std  = {std:.6f}")

    return mean, std


# ============================================================
# DataLoader
# ============================================================

def create_dataloader(
    train_dataset,
    val_dataset,
    test_dataset,
    batch_size=DEFAULT_BATCH_SIZE,
    num_workers=DEFAULT_NUM_WORKERS,
    pin_memory=DEFAULT_PIN_MEMORY,
):

    def make(ds, shuffle):
        return DataLoader(
            ds,
            batch_size=batch_size,
            shuffle=shuffle,
            num_workers=num_workers,
            pin_memory=pin_memory,
        )

    return make(train_dataset, True), make(val_dataset, False), make(test_dataset, False)


# ============================================================
# Unified Loader
# ============================================================
def load_dataset(
    name,
    root_dir,
    batch_size=DEFAULT_BATCH_SIZE,
    normalize=True,
    norm_type=None,
    **kwargs
):
    """
    Build train / val / test loaders.

    Split (see SPLIT_RATIO / _assign_splits):
      repeat          -> 7:1:1 stratified by class (fixed seed)
      user / position -> test = held-out users / positions,
                         rest split train:val = 7:1 stratified by class
      ut_har          -> the dataset's own X_train / X_val / X_test files

    The best epoch is chosen on val; test is only used for the final report.

    kwargs: case_split, split_mode, shape_option, num_sub (XRF55),
            target_time, intel_shape, allowed_labels.
    """

    name = name.lower()
    shape_option = kwargs.get("shape_option", "2d")
    num_sub = kwargs.get("num_sub", 30) # Used for XRF55
    target_time = kwargs.get("target_time", None)
    intel_shape = kwargs.get("intel_shape", False)
    allowed_labels = kwargs.get("allowed_labels", DEFAULT_SELECTED_LABELS)
    case_split = kwargs.get("case_split", 0)
    split_mode = _check_split_mode(kwargs.get("split_mode", DEFAULT_SPLIT_MODE))
    ratio = ":".join(map(str, SPLIT_RATIO))
    if split_mode == "user":
        print(f"Split mode = user, case = {case_split} (val = 1/8 of train users' samples)")
    elif split_mode == "position":
        if not 0 <= case_split < N_POSITION_CASES:
            raise ValueError(f"split_mode=position needs case_split in 0..{N_POSITION_CASES - 1}")
        print(f"Split mode = position, case = {case_split} "
              f"(test positions: pos % {N_POSITION_CASES} == {case_split}, "
              f"val = 1/8 of the other positions' samples)")
    else:
        print(f"Split mode = repeat (random {ratio} train:val:test, stratified by class, "
              f"seed={SPLIT_SEED})")

    # =======================================================
    # UT_HAR
    # =======================================================
    if name == "ut_har":
        if norm_type is None:
            norm_type = "minmax"

        mean = std = None
        if normalize and norm_type == "zscore":
            mean, std = compute_ut_har_mean_std(
                root_dir
            )

        def make(split):
            return UTHARDataset(
                root_dir=root_dir,
                split=split,
                normalize=normalize,
                norm_type=norm_type,
                mean=mean,
                std=std,
            )

    # =======================================================
    # SSHAR ESP / ASUS
    # =======================================================
    elif name in ("sshar_esp", "sshar_asus"):
        cls = SSHAR_ESP_Dataset if name == "sshar_esp" else SSHAR_ASUS_Dataset
        if norm_type is None:
            norm_type = "zscore"

        common = dict(
            root_dir=root_dir,
            case_split=case_split,
            split_mode=split_mode,
            shape_option=shape_option,
            target_time=target_time,
            intel_shape=intel_shape,
            allowed_labels=allowed_labels,
        )

        mean = std = None
        if normalize and norm_type == "zscore":
            # mean/std from the TRAIN split only
            mean, std = compute_sshar_mean_std(cls(split="train", normalize=False, **common))

        def make(split):
            return cls(
                split=split,
                normalize=normalize,
                norm_type=norm_type,
                mean=mean,
                std=std,
                **common,
            )

    # =======================================================
    # XRF55
    # =======================================================
    elif name == "xrf55":
        if norm_type is None:
            norm_type = "zscore"

        mean = std = None
        if normalize and norm_type == "zscore":
            mean, std = compute_xrf55_mean_std(root_dir, case_split, shape_option, num_sub, target_time,
                                               split_mode=split_mode)

        def make(split):
            return XRF55Dataset(
                root_dir=root_dir,
                split=split,
                case_split=case_split,
                split_mode=split_mode,
                shape_option=shape_option,
                num_sub=num_sub,
                normalize=normalize,
                norm_type=norm_type,
                mean=mean,
                std=std,
                target_time=target_time,
                allowed_labels=allowed_labels,
            )

    else:
        raise ValueError(
            f"Unknown dataset: {name}"
        )

    train_dataset = make("train")
    val_dataset = make("val")
    test_dataset = make("test")

    # =======================================================
    for split_name, ds in (("train", train_dataset), ("val", val_dataset), ("test", test_dataset)):
        if len(ds) == 0:
            raise RuntimeError(
                f"[{name}] {split_name} split is empty (0 samples). "
                f"Check --data_root='{root_dir}' (files must exist under it, "
                f"named as expected), the allowed labels {tuple(allowed_labels)}, "
                f"and the split mode (split_mode={split_mode}, "
                f"case_split={case_split})."
            )

    train_loader, val_loader, test_loader = create_dataloader(
        train_dataset,
        val_dataset,
        test_dataset,
        batch_size=batch_size,
    )

    # Handling info extraction
    x, _ = train_dataset[0]
    input_shape = tuple(x.shape)

    num_classes = len(set(train_dataset.y)) if isinstance(train_dataset, UTHARDataset) else len(set(s["label"] for s in train_dataset.samples))

    info = {
        "dataset": name,
        "input_shape": input_shape,
        "num_classes": num_classes,
        "train_size": len(train_dataset),
        "val_size": len(val_dataset),
        "test_size": len(test_dataset),
    }

    return (
        train_loader,
        val_loader,
        test_loader,
        info,
    )


In [ ]:
%%writefile /kaggle/working/cmr_har_code/train.py
"""
train.py

Simple training utilities
"""

import inspect
import os
import time
import pandas as pd

import torch
import torch.nn as nn
from torch.optim import lr_scheduler


SCHEDULER_ARG_KEYS = {
    "steplr": {"step_size": "step_size", "gamma": "gamma"},
    "multisteplr": {"milestones": "milestones", "gamma": "gamma"},
    "cosineannealinglr": {"t_max": "T_max", "T_max": "T_max", "eta_min": "eta_min"},
    "reducelronplateau": {"patience": "patience", "threshold": "threshold", "gamma": "factor"},
    "onecyclelr": {
        "max_lr": "max_lr",
        "steps_per_epoch": "steps_per_epoch",
        "total_steps": "total_steps",
        "epochs": "epochs",
    },
}


def create_scheduler(optimizer, scheduler_cfg=None):
    """
    Create a learning-rate scheduler from a config dict.

    scheduler_cfg = {
        "name": "cosineannealinglr",
        "args": {...},              # only keys relevant to `name` are used
        "warmup_epochs": 5,         # optional linear warmup (epoch-level)
    }
    """
    if scheduler_cfg is None:
        return None

    scheduler_name = scheduler_cfg.get("name")
    if scheduler_name is None or scheduler_name.lower() == "none":
        return None
    scheduler_name = scheduler_name.lower()
    if scheduler_name not in SCHEDULER_ARG_KEYS:
        raise ValueError(f"Unsupported scheduler: {scheduler_name}.")

    raw_args = scheduler_cfg.get("args", {}) or {}
    key_map = SCHEDULER_ARG_KEYS[scheduler_name]
    scheduler_args = {
        key_map[key]: value
        for key, value in raw_args.items()
        if key in key_map and value is not None
    }

    if scheduler_name == "steplr":
        scheduler = lr_scheduler.StepLR(optimizer, **scheduler_args)
    elif scheduler_name == "multisteplr":
        milestones = scheduler_args.get("milestones")
        if isinstance(milestones, str):
            scheduler_args["milestones"] = [
                int(x)
                for x in milestones.split(",")
                if x.strip()
            ]
        scheduler = lr_scheduler.MultiStepLR(optimizer, **scheduler_args)
    elif scheduler_name == "cosineannealinglr":
        scheduler = lr_scheduler.CosineAnnealingLR(optimizer, **scheduler_args)
    elif scheduler_name == "reducelronplateau":
        return lr_scheduler.ReduceLROnPlateau(optimizer, **scheduler_args)
    else:
        return lr_scheduler.OneCycleLR(optimizer, **scheduler_args)

    warmup_epochs = scheduler_cfg.get("warmup_epochs") or 0
    if warmup_epochs > 0:
        warmup = lr_scheduler.LinearLR(
            optimizer,
            start_factor=1.0 / (warmup_epochs + 1),
            end_factor=1.0,
            total_iters=warmup_epochs,
        )
        scheduler = lr_scheduler.SequentialLR(
            optimizer,
            schedulers=[warmup, scheduler],
            milestones=[warmup_epochs],
        )
    return scheduler


def create_optimizer(model, lr, optimizer_cfg=None):
    """Create optimizer. optimizer_cfg = {"name": "adam"|"adamw", "weight_decay": float}."""
    optimizer_cfg = optimizer_cfg or {}
    name = (optimizer_cfg.get("name") or "adam").lower()
    weight_decay = optimizer_cfg.get("weight_decay") or 0.0
    if name == "adam":
        return torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    if name == "adamw":
        return torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    raise ValueError(f"Unsupported optimizer: {name}.")


# ============================================================
# Train one epoch
# ============================================================
def train_one_epoch(
    model,
    loader,
    criterion,
    optimizer,
    device,
):
    """
    Train one epoch.

    Returns
    -------
    train_loss
    train_acc
    """

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for x, y in loader:
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        output = model(x)
        loss = criterion(output, y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        pred = output.argmax(dim=1)
        correct += (pred == y).sum().item()
        total += y.size(0)

    train_loss = running_loss / len(loader)
    train_acc = correct / total

    return train_loss, train_acc

# ============================================================
# Evaluate
# ============================================================
@torch.no_grad()
def evaluate(
    model,
    loader,
    criterion,
    device,
):
    """
    Evaluate model.

    Returns
    -------
    loss
    accuracy
    y_true
    y_pred
    """

    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    y_true = []
    y_pred = []

    for x, y in loader:
        x = x.to(device)
        y = y.to(device)

        output = model(x)
        loss = criterion(output, y)

        running_loss += loss.item()
        pred = output.argmax(dim=1)
        correct += (pred == y).sum().item()
        total += y.size(0)

        y_true.extend(
            y.cpu().numpy()
        )

        y_pred.extend(
            pred.cpu().numpy()
        )

    test_loss = running_loss / len(loader)
    test_acc = correct / total

    return (
        test_loss,
        test_acc,
        y_true,
        y_pred,
    )

# ============================================================
# Train
# ============================================================

def train(
    model,
    train_loader,
    val_loader,
    test_loader,
    epochs,
    lr,
    device,
    save_dir,
    scheduler_cfg=None,
    optimizer_cfg=None,
    patience=10,
):
    """
    Train model.

    Parameters
    ----------
    scheduler_cfg : dict or None
        Optional scheduler configuration dict with keys:
        - name: scheduler type
        - args: scheduler arguments
        - warmup_epochs: optional linear warmup
    optimizer_cfg : dict or None
        {"name": "adam" | "adamw", "weight_decay": float}
    patience : int
        Early stopping: stop when val acc has not improved for `patience`
        epochs in a row. 0 or None disables it.

    Returns
    -------
    history
    y_true
    y_pred
    """

    os.makedirs(save_dir, exist_ok=True)

    model = model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = create_optimizer(model, lr, optimizer_cfg)
    scheduler = create_scheduler(optimizer, scheduler_cfg)

    history = []

    best_acc = -1.0  # ensure best_model.pth is saved at least once
    best_epoch = 0
    epochs_no_improve = 0

    start_time = time.time()
    # --------------------------------------------------------
    for epoch in range(1, epochs + 1):
        train_loss, train_acc = train_one_epoch(
            model=model,
            loader=train_loader,
            criterion=criterion,
            optimizer=optimizer,
            device=device,
        )

        val_loss, val_acc, _, _ = evaluate(
            model=model,
            loader=val_loader,
            criterion=criterion,
            device=device,
        )

        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val_loss,
            "val_acc": val_acc,
            "lr": optimizer.param_groups[0]["lr"],
        })

        print(
            f"[{epoch:03d}/{epochs}] "
            f"Train Loss: {train_loss:.4f}  "
            f"Train Acc: {train_acc:.4f}  "
            f"Val Loss: {val_loss:.4f}  "
            f"Val Acc: {val_acc:.4f}"
        )

        # ----------------------------------------------------
        # Save best model (selected on VAL, never on test)
        # ----------------------------------------------------
        if val_acc > best_acc:
            best_acc = val_acc
            best_epoch = epoch
            epochs_no_improve = 0
            torch.save(
                model.state_dict(),
                os.path.join(
                    save_dir,
                    "best_model.pth",
                ),
            )

        if scheduler is not None:
            if isinstance(
                scheduler,
                lr_scheduler.ReduceLROnPlateau,
            ):
                scheduler.step(val_loss)
            else:
                scheduler.step()

        # ----------------------------------------------------
        # Early stopping (on val acc)
        # ----------------------------------------------------
        if epoch != best_epoch:
            epochs_no_improve += 1
        if patience and epochs_no_improve >= patience:
            print(
                f"Early stopping at epoch {epoch}: val acc has not improved "
                f"for {patience} epochs (best {best_acc:.4f} @ epoch {best_epoch})"
            )
            break

    # --------------------------------------------------------
    # Training finished
    # --------------------------------------------------------
    elapsed = time.time() - start_time

    print("\nTraining Finished")
    print(f"Best Epoch : {best_epoch}")
    print(f"Best Val Acc: {best_acc:.4f}")
    print(f"Time       : {elapsed:.1f} sec")

    # --------------------------------------------------------
    # Save history
    # --------------------------------------------------------
    history = pd.DataFrame(history)
    history.to_csv(
        os.path.join(
            save_dir,
            "history.csv",
        ),
        index=False,
    )

    # --------------------------------------------------------
    # Load best model
    # --------------------------------------------------------
    model.load_state_dict(
        torch.load(
            os.path.join(
                save_dir,
                "best_model.pth",
            ),
            map_location=device,
        )
    )

    # --------------------------------------------------------
    # Final evaluation on TEST (best-on-val checkpoint)
    # --------------------------------------------------------
    test_loss, test_acc, y_true, y_pred = evaluate(
        model=model,
        loader=test_loader,
        criterion=criterion,
        device=device,
    )
    print(f"Test Acc   : {test_acc:.4f}")

    return (
        history,
        y_true,
        y_pred,
        best_epoch,
        best_acc,
    )

In [ ]:
%%writefile /kaggle/working/cmr_har_code/metrics.py
"""
metrics.py

Evaluation utilities
"""

import json
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch

try:
    from thop import profile
except ImportError:
    profile = None

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)


def measure_model_complexity(model, input_shape):
    """Measure MACs and FLOPs for one inference sample."""

    result = {
        "macs": None,
        "flops": None,
        "macs_per_sample": None,
        "flops_per_sample": None,
    }

    if profile is None:
        print("Warning: install thop to measure MACs/FLOPs.")
        return result

    was_training = model.training
    device = next(model.parameters()).device
    dummy_input = torch.zeros(
        (1, *input_shape),
        device=device,
    )

    try:
        model.eval()
        with torch.no_grad():
            macs, _ = profile(
                model,
                inputs=(dummy_input,),
                verbose=False,
            )
        flops = 2 * macs
        result.update({
            "macs": int(macs),
            "flops": int(flops),
            "macs_per_sample": int(macs),
            "flops_per_sample": int(flops),
        })
    except Exception as exc:
        print(
            "Warning: could not measure MACs/FLOPs "
            f"for this model: {exc}"
        )
    finally:
        model.train(was_training)

    return result


# ============================================================
# Compute Metrics
# ============================================================
def compute_metrics(y_true, y_pred):
    """
    Compute classification metrics.

    Returns
    -------
    metrics : dict
    cm : ndarray
    report : dict
    """

    metrics = {
        "accuracy": accuracy_score(
            y_true,
            y_pred,
        ),
        "precision_macro": precision_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0,
        ),

        "recall_macro": recall_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0,
        ),

        "f1_macro": f1_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0,
        ),

        "precision_weighted": precision_score(
            y_true,
            y_pred,
            average="weighted",
            zero_division=0,
        ),

        "recall_weighted": recall_score(
            y_true,
            y_pred,
            average="weighted",
            zero_division=0,
        ),

        "f1_weighted": f1_score(
            y_true,
            y_pred,
            average="weighted",
            zero_division=0,
        ),
    }

    cm = confusion_matrix(
        y_true,
        y_pred,
    )

    report = classification_report(
        y_true,
        y_pred,
        output_dict=True,
        zero_division=0,
    )

    return metrics, cm, report


# ============================================================
# Save metrics.json
# ============================================================
def save_metrics(
    save_dir,
    metrics,
    dataset,
    model,
    best_epoch,
    train_size,
    test_size,
    training_time,
    total_params=None,
    trainable_params=None,
    complexity=None,
    val_size=None,
    best_val_acc=None,
    epochs_trained=None,
):

    data = {
        "dataset": dataset,
        "model": model,
        "accuracy":
            float(metrics["accuracy"]),
        "precision_macro":
            float(metrics["precision_macro"]),
        "recall_macro":
            float(metrics["recall_macro"]),
        "f1_macro":
            float(metrics["f1_macro"]),
        "precision_weighted":
            float(metrics["precision_weighted"]),
        "recall_weighted":
            float(metrics["recall_weighted"]),
        "f1_weighted":
            float(metrics["f1_weighted"]),
        "best_epoch":
            int(best_epoch),
        "epochs_trained":
            None if epochs_trained is None else int(epochs_trained),
        "train_size":
            int(train_size),
        "val_size":
            None if val_size is None else int(val_size),
        "test_size":
            int(test_size),
        "best_val_acc":
            None if best_val_acc is None else float(best_val_acc),
        "training_time":
            float(training_time),
        "total_params":
            total_params,
        "trainable_params":
            trainable_params,
        "macs":
            None if complexity is None else complexity.get("macs"),
        "flops":
            None if complexity is None else complexity.get("flops"),
        "macs_per_sample":
            None if complexity is None else complexity.get("macs_per_sample"),
        "flops_per_sample":
            None if complexity is None else complexity.get("flops_per_sample"),
    }

    with open(
        os.path.join(
            save_dir,
            "metrics.json",
        ),
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            data,
            f,
            indent=4,
        )


def save_model_complexity(
    save_dir,
    dataset,
    model,
    total_params,
    trainable_params,
    complexity,
):
    """Save model complexity before the potentially long training run."""

    data = {
        "dataset": dataset,
        "model": model,
        "status": "pending_training",
        "total_params": total_params,
        "trainable_params": trainable_params,
        "macs": complexity.get("macs"),
        "flops": complexity.get("flops"),
        "macs_per_sample": complexity.get("macs_per_sample"),
        "flops_per_sample": complexity.get("flops_per_sample"),
    }

    with open(
        os.path.join(save_dir, "metrics.json"),
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(data, f, indent=4)


# ============================================================
# Save predictions
# ============================================================
def save_predictions(
    save_dir,
    y_true,
    y_pred,
):

    df = pd.DataFrame({
        "sample":
            np.arange(
                len(y_true)
            ),
        "true":
            y_true,
        "pred":
            y_pred,
    })

    df.to_csv(
        os.path.join(
            save_dir,
            "predictions.csv",
        ),
        index=False,

    )


# ============================================================
# Save classification report
# ============================================================
def save_classification_report(
    save_dir,
    report,
):

    df = pd.DataFrame(
        report
    ).transpose()

    df.to_csv(
        os.path.join(
            save_dir,
            "classification_report.csv",
        ),

        index=True,

    )


# ============================================================
# Save confusion matrix csv
# ============================================================
def save_confusion_matrix(
    save_dir,
    cm,
    class_names=None,
):

    if class_names is None:
        class_names = [
            str(i)
            for i in range(
                len(cm)
            )
        ]

    df = pd.DataFrame(
        cm,
        index=class_names,
        columns=class_names,
    )

    df.to_csv(
        os.path.join(
            save_dir,
            "confusion_matrix.csv",
        )

    )

    # ============================================================
# Learning Curve
# ============================================================
def plot_learning_curve(
    history,
    save_dir,
):
    """
    Plot learning curves.

    Parameters
    ----------
    history
        pandas.DataFrame
        hoặc history.csv
    """

    if isinstance(history, str):
        history = pd.read_csv(history)

    # new runs log "val_*"; old history.csv files only have "test_*"
    ev, ev_label = ("val", "Val") if "val_loss" in history else ("test", "Test")

    plt.figure(figsize=(10, 4))
    # --------------------------------------------------------
    # Loss
    # --------------------------------------------------------
    plt.subplot(1, 2, 1)

    plt.plot(
        history["epoch"],
        history["train_loss"],
        label="Train",
        linewidth=2,
    )

    plt.plot(
        history["epoch"],
        history[f"{ev}_loss"],
        label=ev_label,
        linewidth=2,
    )

    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Loss")
    plt.grid(True)
    plt.legend()

    # --------------------------------------------------------
    # Accuracy
    # --------------------------------------------------------
    plt.subplot(1, 2, 2)
    plt.plot(
        history["epoch"],
        history["train_acc"],
        label="Train",
        linewidth=2,
    )

    plt.plot(
        history["epoch"],
        history[f"{ev}_acc"],
        label=ev_label,
        linewidth=2,
    )

    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title("Accuracy")
    plt.grid(True)
    plt.legend()

    plt.tight_layout()
    plt.savefig(
        os.path.join(
            save_dir,
            "learning_curve.png",
        ),
        dpi=300,
    )
    plt.close()


# ============================================================
# Plot Confusion Matrix
# ============================================================
def plot_confusion_matrix(
    save_dir,
    cm,
    class_names=None,
):

    if class_names is None:
        class_names = [
            str(i)
            for i in range(
                len(cm)
            )
        ]

    plt.figure(figsize=(7, 6))

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=class_names,
        yticklabels=class_names,
    )

    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title("Confusion Matrix")
    plt.tight_layout()
    plt.savefig(
        os.path.join(
            save_dir,
            "confusion_matrix.png",
        ),
        dpi=300,
    )

    plt.close()


# ============================================================
# Complete Evaluation
# ============================================================

def evaluate_model(
    history,
    y_true,
    y_pred,
    save_dir,
    dataset,
    model,
    best_epoch,
    train_size,
    test_size,
    training_time,
    class_names=None,
    total_params=None,
    trainable_params=None,
    complexity=None,
    val_size=None,
    best_val_acc=None,
    epochs_trained=None,
):
    """
    Save all evaluation results.

    Output
    ------
    history.csv                  (đã lưu ở train.py)
    metrics.json
    predictions.csv
    classification_report.csv
    confusion_matrix.csv
    confusion_matrix.png
    learning_curve.png
    """

    os.makedirs(
        save_dir,
        exist_ok=True,
    )
    # --------------------------------------------------------
    # Metrics
    # --------------------------------------------------------
    metrics, cm, report = compute_metrics(
        y_true,
        y_pred,
    )
    # --------------------------------------------------------
    # Save files
    # --------------------------------------------------------
    save_metrics(
        save_dir=save_dir,
        metrics=metrics,
        dataset=dataset,
        model=model,
        best_epoch=best_epoch,
        train_size=train_size,
        test_size=test_size,
        training_time=training_time,
        total_params=total_params,
        trainable_params=trainable_params,
        complexity=complexity,
        val_size=val_size,
        best_val_acc=best_val_acc,
        epochs_trained=epochs_trained,
    )

    save_predictions(
        save_dir,
        y_true,
        y_pred,
    )

    save_classification_report(
        save_dir,
        report,
    )

    save_confusion_matrix(
        save_dir,
        cm,
        class_names,
    )

    plot_confusion_matrix(
        save_dir,
        cm,
        class_names,
    )

    plot_learning_curve(
        history,
        save_dir,
    )

    # --------------------------------------------------------
    # Print result
    # --------------------------------------------------------

    print("\n" + "=" * 60)
    print("Evaluation Result")
    print("=" * 60)
    print(f"Accuracy          : {metrics['accuracy']:.4f}")
    print(f"Macro Precision   : {metrics['precision_macro']:.4f}")
    print(f"Macro Recall      : {metrics['recall_macro']:.4f}")
    print(f"Macro F1-score    : {metrics['f1_macro']:.4f}")
    print("=" * 60)
    return metrics

In [ ]:
%%writefile /kaggle/working/cmr_har_code/main.py
"""
main.py

WiFi HAR Benchmark
"""

import argparse
import json
import os
import random
import time

import numpy as np

import torch

from datasets import load_dataset
from models import get_model
from train import train
from metrics import (
    evaluate_model,
    measure_model_complexity,
    save_model_complexity,
)

# ============================================================
# Global seed
# ============================================================
SEED = 42

def set_seed(seed=SEED):
    """
    Set global random seed for reproducibility.

    Covers
    ------
    - Python `random`
    - NumPy
    - PyTorch (CPU + CUDA)
    - cuDNN deterministic mode
    """

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# ============================================================
# Argument Parser
# ============================================================
def parse_args():
    parser = argparse.ArgumentParser(
        description="WiFi HAR Benchmark"
    )

    parser.add_argument(
        "--dataset",
        type=str,
        required=True,
        choices=[
            "ut_har",
            "xrf55",
            "sshar_esp",
            "sshar_asus",
        ],
        help="Dataset name",
    )

    parser.add_argument(
        "--model",
        type=str,
        required=True,
        choices=[
            "mlp",
            "cnn5",
            "bilstm",
            "vit",
            "resnet18",
            "resnet1d",
            "cnn_gru",
            "wiformer",
        ],
        help="Model name",
    )

    parser.add_argument(
        "--data_root",
        type=str,
        required=True,
        help="Dataset root directory",
    )

    parser.add_argument(
        "--result_root",
        type=str,
        default="results",
        help="Directory to save results",
    )

    parser.add_argument(
        "--epochs",
        type=int,
        default=100,
    )

    parser.add_argument(
        "--patience",
        type=int,
        default=10,
        help="Early stopping: stop after N epochs without val-acc improvement (0 = off)",
    )

    parser.add_argument(
        "--batch_size",
        type=int,
        default=None,
        help="Default: per-model (wiformer=32, others=64)",
    )

    parser.add_argument(
        "--lr",
        type=float,
        default=None,
        help="Default: 1e-3 for all models",
    )

    parser.add_argument(
        "--optimizer",
        type=str,
        default=None,
        choices=["adam", "adamw"],
        help="Default: adam for all models",
    )

    parser.add_argument(
        "--weight_decay",
        type=float,
        default=None,
        help="Default: 0 for all models",
    )

    parser.add_argument(
        "--warmup_epochs",
        type=int,
        default=None,
        help="Linear warmup epochs before the scheduler (wiformer=5)",
    )

    parser.add_argument(
        "--scheduler",
        type=str,
        default="none",
        choices=[
            "none",
            "steplr",
            "multisteplr",
            "cosineannealinglr",
            "reducelronplateau",
            "onecyclelr",
        ],
        help="Optional learning rate scheduler",
    )

    parser.add_argument(
        "--scheduler-step-size",
        type=int,
        default=30,
        help="Step size for StepLR",
    )

    parser.add_argument(
        "--scheduler-milestones",
        type=str,
        default="",
        help="Comma-separated milestones for MultiStepLR",
    )

    parser.add_argument(
        "--scheduler-gamma",
        type=float,
        default=0.1,
        help="Gamma for scheduler decay",
    )

    parser.add_argument(
        "--scheduler-t-max",
        type=int,
        default=None,
        help="T_max for CosineAnnealingLR (default: epochs - warmup)",
    )

    parser.add_argument(
        "--scheduler-eta-min",
        type=float,
        default=0.0,
        help="Eta min for CosineAnnealingLR",
    )

    parser.add_argument(
        "--scheduler-patience",
        type=int,
        default=10,
        help="Patience for ReduceLROnPlateau",
    )

    parser.add_argument(
        "--scheduler-threshold",
        type=float,
        default=1e-4,
        help="Threshold for ReduceLROnPlateau",
    )

    parser.add_argument(
        "--scheduler-max-lr",
        type=float,
        default=None,
        help="Max LR for OneCycleLR",
    )

    parser.add_argument(
        "--scheduler-steps-per-epoch",
        type=int,
        default=None,
        help="Steps per epoch for OneCycleLR",
    )

    parser.add_argument(
        "--scheduler-total-steps",
        type=int,
        default=None,
        help="Total steps for OneCycleLR",
    )

    parser.add_argument(
        "--device",
        type=str,
        default="cuda",
        choices=[
            "cpu",
            "cuda",
        ],
    )

    parser.add_argument(
        "--k2_layers",
        type=int,
        default=2,
        help="WiFormer: number of layers in Block 2 (Rx x Sub attention)",
    )

    parser.add_argument(
        "--k3_layers",
        type=int,
        default=4,
        help="WiFormer: number of layers in Block 3 (temporal Transformer)",
    )

    parser.add_argument(
        "--k1_branch",
        type=str,
        default="both",
        choices=["both", "time", "freq", "none"],
        help="WiFormer Block 1 branches: Conv1d+FFT (both), Conv1d only (time), FFT only (freq), no Block 1 = Linear(W, D) on the raw window (none)",
    )

    # ---- WiFormer ablation knobs (defaults = full model) ----
    parser.add_argument(
        "--k2_mode",
        type=str,
        default="joint",
        choices=["joint", "per_rx", "pool"],
        help="WiFormer Block 2: joint Rx x Sub attention / attention per Rx then mean / mean pooling",
    )
    parser.add_argument(
        "--k2_pe",
        type=str,
        default="sub",
        choices=["sub", "rx_sub", "none"],
        help="WiFormer Block 2 positional embedding",
    )
    parser.add_argument(
        "--k3_pos",
        type=str,
        default="alibi",
        choices=["alibi", "learned", "none"],
        help="WiFormer Block 3 positional encoding (k3_layers=0 -> mean pooling over windows)",
    )
    parser.add_argument(
        "--ant_mode",
        type=str,
        default="first",
        choices=["first", "all"],
        help="WiFormer: keep antenna[0] of each Rx, or use every (rx, antenna) as a view",
    )
    parser.add_argument(
        "--no_sample_norm",
        action="store_true",
        help="WiFormer: disable per-sample z-score",
    )
    # ---- WiFormer capacity (defaults = current model) ----
    parser.add_argument("--d_model", type=int, default=64, help="WiFormer: token dim D (all blocks)")
    parser.add_argument("--feat", type=int, default=64, help="WiFormer Block 1: Conv1d output channels")
    parser.add_argument("--k1_conv", type=int, default=2, choices=[2, 3],
                        help="WiFormer Block 1: number of Conv1d layers")
    parser.add_argument("--k1_pool", type=str, default="avg",
                        help="WiFormer Block 1: 'avg' (global avg) or 'flatN' (keep N steps, e.g. flat4)")
    parser.add_argument("--k2_heads", type=int, default=4, help="WiFormer Block 2: attention heads")
    parser.add_argument("--k2_ff", type=int, default=None, help="WiFormer Block 2: FFN width (default 4*D)")
    parser.add_argument("--k3_heads", type=int, default=4, help="WiFormer Block 3: attention heads")
    parser.add_argument("--k3_ff", type=int, default=None, help="WiFormer Block 3: FFN width (default 4*D)")
    parser.add_argument("--dropout", type=float, default=0.1, help="WiFormer: dropout")
    parser.add_argument("--window", type=int, default=64, help="WiFormer window W")
    parser.add_argument("--stride", type=int, default=32, help="WiFormer stride")
    parser.add_argument(
        "--run_name",
        type=str,
        default="",
        help="Suffix of the result folder: <dataset>_<model>[_<run_name>]",
    )

    parser.add_argument(
            "--case_split",
            type=int,
            default=0,
            choices=[0, 1, 2, 3, 4],
            help="case split: 0..4 for --split_mode=user, 0..3 for --split_mode=position"
        )

    parser.add_argument(
            "--split_mode",
            type=str,
            default="repeat",
            choices=["repeat", "user", "position"],
            help="split: 'repeat' (random 7:1:1 train:val:test, stratified by class), 'user' (cross-user, uses --case_split) "
                 "or 'position' (SSHAR cross-position, --case_split 0..3 holds out pos %% 4 == case)"
        )

    return parser.parse_args()


def get_default_train_cfg(model_name):
    """Per-model defaults for lr / batch size / optimizer."""
    model_name = model_name.lower()
    if model_name == "wiformer":
        # Adam, lr=1e-3 (as in common WiFi-HAR baselines). Warmup + cosine
        # are kept for the Transformer so the large initial lr stays stable.
        return {
            "lr": 1e-3,
            "batch_size": 32,
            "optimizer": "adam",
            "weight_decay": 0.0,
            "warmup_epochs": 5,
        }
    return {
        "lr": 1e-3,
        "batch_size": 64,
        "optimizer": "adam",
        "weight_decay": 0.0,
        "warmup_epochs": 0,
    }


def get_default_scheduler_cfg(model_name, epochs=100, warmup_epochs=0):
    model_name = model_name.lower()
    if model_name == "wiformer":
        return {
            "name": "cosineannealinglr",
            "args": {
                "t_max": max(1, epochs - warmup_epochs),
            },
        }
    if model_name == "resnet1d":
        return {
            "name": "multisteplr",
            "args": {
                "milestones": [40, 80, 120, 160],
                "gamma": 0.5,
            },
        }
    if model_name == "resnet18":
        return {
            "name": "steplr",
            "args": {
                "step_size": 30,
                "gamma": 0.1,
            },
        }
    return None


# ============================================================
# Main
# ============================================================

def main():
    args = parse_args()
    train_defaults = get_default_train_cfg(args.model)
    for key, value in train_defaults.items():
        if getattr(args, key) is None:
            setattr(args, key, value)
    # --------------------------------------------------------
    # Reproducibility
    # --------------------------------------------------------
    set_seed(SEED)
    # --------------------------------------------------------
    # Device
    # --------------------------------------------------------
    if args.device == "cuda" and torch.cuda.is_available():
        device = torch.device("cuda")
    else:
        device = torch.device("cpu")

    print("=" * 60)
    print("WiFi HAR Benchmark")
    print("=" * 60)
    print("Dataset :", args.dataset)
    print("Model   :", args.model)
    print("Device  :", device)
    print()

    # --------------------------------------------------------
    # Load dataset
    # --------------------------------------------------------
    dataset_kwargs = {}
    dataset_kwargs["case_split"] = args.case_split
    dataset_kwargs["split_mode"] = args.split_mode

    if args.model == "cnn_gru":
        train_loader, val_loader, test_loader, info = load_dataset(
            name=args.dataset,
            root_dir=args.data_root,
            batch_size=args.batch_size,
            shape_option="3d",
            intel_shape=True,
            **dataset_kwargs,
        )
    else:
        train_loader, val_loader, test_loader, info = load_dataset(
            name=args.dataset,
            root_dir=args.data_root,
            batch_size=args.batch_size,
            intel_shape=True,
            **dataset_kwargs,
        )

    input_shape = info["input_shape"]
    num_classes = info["num_classes"]
    train_size = info["train_size"]
    val_size = info["val_size"]
    test_size = info["test_size"]

    print("Input Shape :", input_shape)
    print("Classes     :", num_classes)
    print("Train Size  :", train_size)
    print("Val Size    :", val_size)
    print("Test Size   :", test_size)
    print()

    # --------------------------------------------------------
    # Build model
    # --------------------------------------------------------
    model_kwargs = {}
    if args.model == "wiformer":
        model_kwargs.update(
            k2_layers=args.k2_layers,
            k3_layers=args.k3_layers,
            k1_branch=args.k1_branch,
            k2_mode=args.k2_mode,
            k2_pe=args.k2_pe,
            k3_pos=args.k3_pos,
            ant_mode=args.ant_mode,
            sample_norm=not args.no_sample_norm,
            window=args.window,
            stride=args.stride,
            d_model=args.d_model,
            feat=args.feat,
            k1_conv=args.k1_conv,
            k1_pool=args.k1_pool,
            k2_heads=args.k2_heads,
            k2_ff=args.k2_ff,
            k3_heads=args.k3_heads,
            k3_ff=args.k3_ff,
            dropout=args.dropout,
        )

    model = get_model(
        model_name=args.model,
        input_shape=input_shape,
        num_classes=num_classes,
        dataset=args.dataset,
        **model_kwargs,
    )
    model = model.to(device)

    scheduler_cfg = get_default_scheduler_cfg(
        args.model, args.epochs, args.warmup_epochs
    )
    if args.scheduler and args.scheduler.lower() != "none":
        scheduler_cfg = {
            "name": args.scheduler,
            "args": {
                "step_size": args.scheduler_step_size,
                "gamma": args.scheduler_gamma,
                "milestones": args.scheduler_milestones,
                "t_max": (
                    args.scheduler_t_max
                    if args.scheduler_t_max is not None
                    else max(1, args.epochs - args.warmup_epochs)
                ),
                "eta_min": args.scheduler_eta_min,
                "patience": args.scheduler_patience,
                "threshold": args.scheduler_threshold,
                "max_lr": args.scheduler_max_lr,
                "steps_per_epoch": args.scheduler_steps_per_epoch,
                "total_steps": args.scheduler_total_steps,
            },
        }

    if scheduler_cfg is not None:
        scheduler_cfg["warmup_epochs"] = args.warmup_epochs
    optimizer_cfg = {
        "name": args.optimizer,
        "weight_decay": args.weight_decay,
    }
    print(
        f"Optimizer : {args.optimizer} (lr={args.lr}, wd={args.weight_decay})  "
        f"Batch: {args.batch_size}  "
        f"Scheduler: {scheduler_cfg['name'] if scheduler_cfg else 'none'} "
        f"(warmup={args.warmup_epochs})"
    )

    # --------------------------------------------------------
    # Result folder
    # --------------------------------------------------------
    run_dir = f"{args.dataset}_{args.model}"
    if args.run_name:
        run_dir += f"_{args.run_name}"
    save_dir = os.path.join(
        args.result_root,
        run_dir,
    )
    os.makedirs(
        save_dir,
        exist_ok=True,
    )
    with open(os.path.join(save_dir, "config.json"), "w", encoding="utf-8") as f:
        json.dump(vars(args), f, indent=4)
    # --------------------------------------------------------
    # Count and save model complexity before training
    # --------------------------------------------------------
    total_params = sum(
        p.numel()
        for p in model.parameters()
    )

    trainable_params = sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )

    complexity = measure_model_complexity(
        model=model,
        input_shape=input_shape,
    )

    def format_count(value):
        if value is None:
            return "N/A"
        if value >= 1_000_000_000:
            return f"{value / 1_000_000_000:.3f}G"
        if value >= 1_000_000:
            return f"{value / 1_000_000:.3f}M"
        if value >= 1_000:
            return f"{value / 1_000:.3f}K"
        return str(value)

    print("Model Complexity")
    print(f"Total Params     : {format_count(total_params)}")
    print(f"Trainable Params : {format_count(trainable_params)}")
    print(f"MACs / sample    : {format_count(complexity['macs_per_sample'])}")
    print(f"FLOPs / sample   : {format_count(complexity['flops_per_sample'])}")

    save_model_complexity(
        save_dir=save_dir,
        dataset=args.dataset,
        model=args.model,
        total_params=total_params,
        trainable_params=trainable_params,
        complexity=complexity,
    )

    # --------------------------------------------------------
    # Train
    # --------------------------------------------------------
    start_time = time.time()
    history, y_true, y_pred, best_epoch, best_acc = train(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        test_loader=test_loader,
        epochs=args.epochs,
        lr=args.lr,
        device=device,
        save_dir=save_dir,
        scheduler_cfg=scheduler_cfg,
        optimizer_cfg=optimizer_cfg,
        patience=args.patience,
    )
    training_time = time.time() - start_time
    # --------------------------------------------------------
    # Evaluation
    # --------------------------------------------------------
    evaluate_model(
        history=history,
        y_true=y_true,
        y_pred=y_pred,
        save_dir=save_dir,
        dataset=args.dataset,
        model=args.model,
        best_epoch=best_epoch,
        train_size=train_size,
        test_size=test_size,
        training_time=training_time,
        total_params=total_params,
        trainable_params=trainable_params,
        complexity=complexity,
        val_size=val_size,
        best_val_acc=best_acc,
        epochs_trained=len(history),
    )

    print()
    print("=" * 60)
    print("Finished")
    print("Result Folder")
    print(save_dir)
    print("=" * 60)

# ============================================================
# Run
# ============================================================
if __name__ == "__main__":
    main()

# python main.py --data_root=".\data\" --dataset="ut_har" --model="mlp" --case_split=0
# python main.py --data_root=".\data\" --dataset="xrf55" --model="wiformer" --split_mode=user --case_split=2

In [ ]:
%%writefile /kaggle/working/cmr_har_code/ablation.py
"""
ablation.py - WiFormer ablation runner

Run every variant (or a subset) through main.py, each one into its own
folder <result_root>/<dataset>_wiformer_<variant>, then collect all
metrics.json into <result_root>/ablation_<dataset>.csv.

Examples
--------
# list variants
python ablation.py --list

# main ablation (remove one block at a time) on xrf55 - default --set core
python ablation.py --data_root=".\\data\\" --dataset=xrf55

# design-choice variants / everything
python ablation.py --data_root=".\\data\\" --dataset=xrf55 --set design
python ablation.py --data_root=".\\data\\" --dataset=xrf55 --set all

# only a few variants, cross-user case 2, 50 epochs
python ablation.py --data_root=".\\data\\" --dataset=sshar_esp ^
    --variants full wo_fft wo_conv --extra --split_mode=user --case_split=2 --epochs=50

# only rebuild the summary table from finished runs
python ablation.py --dataset=xrf55 --summary_only
"""

import argparse
import csv
import json
import os
import subprocess
import sys

# name -> (group, description, extra CLI args for main.py)
# Every variant changes exactly ONE thing w.r.t. "full".

# ---- Main ablation: remove one block at a time ----
# A removed block is replaced by the simplest op that keeps the tensor shape:
#   Block 1 -> Linear(W, D) on the raw window
#   Block 2 -> mean over the Rx x Sub tokens
#   Block 3 -> mean over the windows
CORE_VARIANTS = {
    "full": ("Full", "CSI-WiFormer (all blocks)", []),
    "wo_fft": ("Block 1", "w/o FFT branch (Conv1d only)", ["--k1_branch=time"]),
    "wo_conv": ("Block 1", "w/o Conv1d branch (FFT only)", ["--k1_branch=freq"]),
    "wo_block1": ("Block 1", "w/o Block 1 (Linear(W,D) on raw window)", ["--k1_branch=none"]),
    "wo_block2": ("Block 2", "w/o Block 2 (mean over Rx x Sub)", ["--k2_mode=pool"]),
    "wo_block3": ("Block 3", "w/o Block 3 (mean over windows)", ["--k3_layers=0"]),
}

# ---- Optional: design choices inside each block ----
DESIGN_VARIANTS = {
    # --- Block 2: Rx x Sub attention ---
    "k2_per_rx": ("Block 2", "Attention per Rx (no cross-Rx), mean over Rx", ["--k2_mode=per_rx"]),
    "k2_pe_none": ("Block 2", "No PE_sub", ["--k2_pe=none"]),
    "k2_pe_rx_sub": ("Block 2", "PE_sub + PE_rx", ["--k2_pe=rx_sub"]),

    # --- Block 3: temporal Transformer ---
    "k3_learned": ("Block 3", "Learned absolute PE instead of ALiBi", ["--k3_pos=learned"]),
    "k3_nopos": ("Block 3", "No positional info (no ALiBi)", ["--k3_pos=none"]),

    # --- Depth ---
    "k2_l1": ("Depth", "Block 2: 1 layer", ["--k2_layers=1"]),
    "k3_l2": ("Depth", "Block 3: 2 layers", ["--k3_layers=2"]),

    # --- Input / preprocessing ---
    "no_norm": ("Input", "No per-sample z-score", ["--no_sample_norm"]),
    "all_ant": ("Input", "All antennas as views (R x A)", ["--ant_mode=all"]),
    "win32": ("Input", "Window 32 / stride 16", ["--window=32", "--stride=16"]),
    "win128": ("Input", "Window 128 / stride 64", ["--window=128", "--stride=64"]),
}

# ---- Optional: scale up (one change at a time, then combined) ----
# step cost vs full measured on SSHAR-ESP (3 x 56 x 1000), CPU, batch 4
D128 = ["--d_model=128", "--k2_heads=8", "--k3_heads=8"]
K1_DEEP = ["--k1_conv=3", "--k1_pool=flat4"]
SCALE_VARIANTS = {
    "k1_flat4": ("Block 1", "Conv1d keeps 4 steps (flat4) instead of global avg", ["--k1_pool=flat4"]),   # ~1.1x
    "k1_deep": ("Block 1", "3 Conv1d layers + flat4", K1_DEEP),                                     # ~1.2x
    "k2_l3": ("Block 2", "Block 2: 3 layers", ["--k2_layers=3"]),                                   # ~1.2x
    "k3_l6": ("Block 3", "Block 3: 6 layers", ["--k3_layers=6"]),                                   # ~1.0x
    "d96": ("Width", "D = 96, 6 heads", ["--d_model=96", "--k2_heads=6", "--k3_heads=6"]),          # ~1.3x
    "d128": ("Width", "D = 128, 8 heads", D128),                                                    # ~1.8x
    "d128_k1deep": ("Combined", "D = 128 + Block 1 deep + dropout 0.2",
                    [*D128, "--feat=128", *K1_DEEP, "--dropout=0.2"]),                              # ~2.5x
}

VARIANTS = {**CORE_VARIANTS, **DESIGN_VARIANTS, **SCALE_VARIANTS}
SETS = {
    "core": list(CORE_VARIANTS),
    # "full" is kept so the summary can compute delta_acc
    "design": ["full", *DESIGN_VARIANTS],
    "scale": ["full", *SCALE_VARIANTS],
    "all": list(VARIANTS),
}

METRIC_KEYS = [
    "accuracy", "f1_macro", "precision_macro", "recall_macro",
    "best_epoch", "epochs_trained", "total_params", "flops_per_sample", "training_time",
]


def parse_args():
    p = argparse.ArgumentParser(description="WiFormer ablation runner")
    p.add_argument("--dataset", type=str, default=None,
                   choices=["ut_har", "xrf55", "sshar_esp", "sshar_asus"])
    p.add_argument("--data_root", type=str, default=None)
    p.add_argument("--result_root", type=str, default="results/ablation")
    p.add_argument("--set", type=str, default="core", choices=list(SETS),
                   help="core = remove one block at a time (default), "
                        "design = design choices inside blocks, "
                        "scale = bigger blocks, all = everything")
    p.add_argument("--variants", nargs="+", default=None,
                   help="Explicit list of variants (overrides --set)")
    p.add_argument("--skip_done", action="store_true",
                   help="Skip variants whose metrics.json already has accuracy")
    p.add_argument("--summary_only", action="store_true")
    p.add_argument("--list", action="store_true")
    p.add_argument("--extra", nargs=argparse.REMAINDER, default=[],
                   help="Everything after --extra is passed to main.py (e.g. --epochs=50)")
    return p.parse_args()


def run_dir(result_root, dataset, name):
    return os.path.join(result_root, f"{dataset}_wiformer_{name}")


def is_done(path):
    f = os.path.join(path, "metrics.json")
    if not os.path.isfile(f):
        return False
    with open(f, encoding="utf-8") as fh:
        return "accuracy" in json.load(fh)


def run_variant(args, name):
    cmd = [
        sys.executable, "main.py",
        f"--data_root={args.data_root}",
        f"--dataset={args.dataset}",
        "--model=wiformer",
        f"--result_root={args.result_root}",
        f"--run_name={name}",
        *VARIANTS[name][2],
        *args.extra,
    ]
    print("\n" + "#" * 70)
    print(f"# [{name}] {VARIANTS[name][1]}")
    print("# " + " ".join(cmd))
    print("#" * 70, flush=True)
    return subprocess.run(cmd).returncode


def summarize(result_root, dataset, names, tag):
    rows = []
    full_acc = None
    for name in names:
        f = os.path.join(run_dir(result_root, dataset, name), "metrics.json")
        if not os.path.isfile(f):
            continue
        with open(f, encoding="utf-8") as fh:
            m = json.load(fh)
        if "accuracy" not in m:
            continue
        group, desc, _ = VARIANTS[name]
        row = {"variant": name, "group": group, "description": desc}
        row.update({k: m.get(k) for k in METRIC_KEYS})
        rows.append(row)
        if name == "full":
            full_acc = m["accuracy"]

    if not rows:
        print("No finished runs found.")
        return

    for r in rows:
        r["delta_acc"] = None if full_acc is None else r["accuracy"] - full_acc

    out = os.path.join(result_root, f"ablation_{dataset}_{tag}.csv")
    os.makedirs(result_root, exist_ok=True)
    fields = ["variant", "group", "description", *METRIC_KEYS, "delta_acc"]
    with open(out, "w", newline="", encoding="utf-8") as fh:
        w = csv.DictWriter(fh, fieldnames=fields)
        w.writeheader()
        w.writerows(rows)

    print(f"\nAblation summary - {dataset} ({tag})")
    print(f"{'variant':<14}{'group':<9}{'acc':>8}{'d_acc':>8}{'F1':>8}{'params':>10}  description")
    for r in rows:
        d = "" if r["delta_acc"] is None else f"{100 * r['delta_acc']:+.2f}"
        print(
            f"{r['variant']:<14}{r['group']:<9}{100 * r['accuracy']:>8.2f}{d:>8}"
            f"{100 * r['f1_macro']:>8.2f}{r['total_params'] or 0:>10}  {r['description']}"
        )
    print(f"\nSaved: {out}")


def main():
    args = parse_args()

    if args.list:
        for title, table in (("core (remove one block at a time)", CORE_VARIANTS),
                             ("design (choices inside blocks)", DESIGN_VARIANTS),
                             ("scale (bigger blocks)", SCALE_VARIANTS)):
            print(f"[{title}]")
            for name, (group, desc, cli) in table.items():
                print(f"  {name:<14}{group:<9}{desc:<48}{' '.join(cli)}")
        return

    if args.dataset is None:
        sys.exit("--dataset is required")

    names = args.variants or SETS[args.set]
    tag = "custom" if args.variants else args.set
    unknown = [n for n in names if n not in VARIANTS]
    if unknown:
        sys.exit(f"Unknown variants: {unknown}. Use --list.")

    if not args.summary_only:
        if args.data_root is None:
            sys.exit("--data_root is required")
        failed = []
        for name in names:
            if args.skip_done and is_done(run_dir(args.result_root, args.dataset, name)):
                print(f"[skip] {name} already done")
                continue
            if run_variant(args, name) != 0:
                failed.append(name)
        if failed:
            print(f"\nFailed variants: {failed}")

    summarize(args.result_root, args.dataset, names, tag)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/cmr_har_code/models/__init__.py
"""Benchmark model package."""

from .mlp import MLP
from .cnn5 import CNN5
from .resnet18 import ResNet18
from .resnet1d import resnet18_1d
from .bilstm import BiLSTM
from .vit import ViT
from .cnn_gru import CNN_GRU
from .wiformer import WiFormer

SUPPORTED_MODELS = [
    "mlp",
    "cnn5",
    "bilstm",
    "vit",
    "resnet18",
    "resnet1d",
    "cnn_gru",
    "wiformer",
]


def list_models():
    return SUPPORTED_MODELS.copy()


def get_model(model_name, input_shape, num_classes, **kwargs):
    model_name = model_name.lower()
    if model_name == "mlp":
        return MLP(
            input_shape=input_shape,
            num_classes=num_classes,
            **kwargs,
        )

    if model_name == "cnn5":
        return CNN5(
            dataset=kwargs.get("dataset"),
            num_classes=num_classes,
        )

    if model_name == "bilstm":
        return BiLSTM(
            input_shape=input_shape,
            num_classes=num_classes,
            **kwargs,
        )

    if model_name == "vit":
        return ViT(
            input_shape=input_shape,
            num_classes=num_classes,
            **kwargs,
        )

    if model_name == "resnet18":
        return ResNet18(
            input_shape=input_shape,
            num_classes=num_classes,
            **kwargs,
        )

    if model_name == "resnet1d":
        return resnet18_1d(
            inchannel=input_shape[0],
            num_classes=num_classes,
        )

    if model_name == "cnn_gru":
        return CNN_GRU(
            input_shape=input_shape,
            num_classes=num_classes,
            **kwargs,
        )

    if model_name == "wiformer":
        return WiFormer(
            input_shape=input_shape,
            num_classes=num_classes,
            **kwargs,
        )

    raise ValueError(
        f"Unsupported model: {model_name}\n"
        f"Supported models: {SUPPORTED_MODELS}"
    )


In [ ]:
%%writefile /kaggle/working/cmr_har_code/models/base.py
import torch.nn as nn


class BaseModel(nn.Module):
    """
    Base class of all benchmark models.
    """

    def __init__(
        self,
        input_shape,
        num_classes,
        **kwargs,
    ):
        super().__init__()

        self.input_shape = input_shape
        self.num_classes = num_classes

    def forward(self, x):
        raise NotImplementedError


In [ ]:
%%writefile /kaggle/working/cmr_har_code/models/bilstm.py
import torch.nn as nn

from .base import BaseModel


class BiLSTM(BaseModel):
    def __init__(
        self,
        input_shape,
        num_classes,
        hidden_dim=64,
        **kwargs,
    ):
        super().__init__(
            input_shape=input_shape,
            num_classes=num_classes,
        )

        # UT_HAR input is typically (batch, 90, 250)
        # where 90 = feature dim, 250 = time steps.
        input_dim = input_shape[0]

        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            bidirectional=True,
            batch_first=False,
        )

        # Match the architecture you provided, but keep it generic
        # instead of hard-coding 7 output classes.
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        # Dataset format: (B, C, T) = (B, 90, 250)
        # LSTM expects: (T, B, C) = (250, B, 90)
        x = x.permute(2, 0, 1)

        _, (ht, _) = self.lstm(x)

        # ht[-1] -> last direction hidden state, shape: (B, hidden_dim)
        output = self.fc(ht[-1])

        return output


In [ ]:
%%writefile /kaggle/working/cmr_har_code/models/cnn5.py
import torch
import torch.nn as nn


class CNN5(nn.Module):
    def __init__(
        self,
        dataset,
        num_classes,
    ):
        super().__init__()
        dataset = dataset.lower()

        if dataset == "ut_har":
            input_size = (250, 90)
        elif dataset == "sshar_esp":
            input_size = (250, 168)
        elif dataset == "sshar_asus":
            input_size = (250, 672)
        else:
            raise ValueError(
                f"Unknown dataset: {dataset}"
            )

        self.encoder = nn.Sequential(
            nn.Conv2d(
                1,
                32,
                kernel_size=7,
                stride=(3, 1),
            ),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(
                32,
                64,
                kernel_size=(5, 4),
                stride=(2, 2),
                padding=(1, 0),
            ),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(
                64,
                96,
                kernel_size=3,
            ),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        with torch.no_grad():
            dummy = torch.zeros(
                1,
                1,
                input_size[0],
                input_size[1],
            )
            dummy = self.encoder(dummy)
            feature_dim = dummy.numel()

        print(
            "LeNet feature size:",
            feature_dim,
        )

        self.fc = nn.Sequential(
            nn.Linear(
                feature_dim,
                128,
            ),
            nn.ReLU(),
            nn.Linear(
                128,
                num_classes,
            ),
        )

    def forward(
        self,
        x,
    ):
        x = x.permute(
            0,
            2,
            1,
        ).unsqueeze(1)

        x = self.encoder(x)
        x = torch.flatten(
            x,
            1,
        )
        x = self.fc(x)

        return x


In [ ]:
%%writefile /kaggle/working/cmr_har_code/models/cnn_gru.py
import torch
import torch.nn as nn

from .base import BaseModel


class CNN_GRU(BaseModel):
    def __init__(
        self,
        input_shape,
        num_classes,
        **kwargs,
    ):
        super().__init__(
            input_shape=input_shape,
            num_classes=num_classes,
        )

        channels, sub, time = input_shape

        self.cnn = nn.Sequential(
            # Conv block 1
            nn.Conv2d(
                channels,
                128,
                kernel_size=5,
                stride=1,
                padding="same",
            ),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.AvgPool2d(
                kernel_size=2,
                stride=2,
            ),
            nn.Dropout(0.6),

            # Conv block 2
            nn.Conv2d(
                128,
                128,
                kernel_size=5,
                stride=1,
                padding="same",
            ),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.AvgPool2d(
                kernel_size=2,
                stride=2,
            ),
        )

        # Pooling không padding:
        # 56 -> 28 -> 14
        # 30 -> 15 -> 7
        sub_out = (sub // 2) // 2

        self.gru = nn.GRU(
            input_size=128 * sub_out,
            hidden_size=256,
            batch_first=True,
        )

        self.fc = nn.Linear(
            256,
            num_classes,
        )

    def forward(self, x):
        # Input:
        # (B, 12, 56, 1000)

        x = self.cnn(x)

        # sub=56:
        # (B, 128, 14, 250)
        #
        # sub=30:
        # (B, 128, 7, 250)

        # TimeDistributed Flatten
        x = x.permute(0, 3, 1, 2)

        # sub=56 -> (B, 250, 1792)
        # sub=30 -> (B, 250, 896)
        x = x.flatten(2)

        x, _ = self.gru(x)

        # Last timestep
        x = x[:, -1]

        return self.fc(x)

In [ ]:
%%writefile /kaggle/working/cmr_har_code/models/mlp.py
import torch
import torch.nn as nn
from .base import BaseModel

class MLP(BaseModel):
    """
    MLP baseline.

    Input
    -----
    (batch, C, T)

    Output
    ------
    (batch, num_classes)
    """

    def __init__(
        self,
        input_shape,
        num_classes,
        **kwargs,
    ):
        super().__init__(
            input_shape=input_shape,
            num_classes=num_classes,
            **kwargs,
        )

        input_dim = input_shape[0] * input_shape[1]
        self.fc = nn.Sequential(
            nn.Linear(
                input_dim,
                1024,
            ),
            nn.ReLU(),
            nn.Linear(
                1024,
                128,
            ),
            nn.ReLU(),
            nn.Linear(
                128,
                num_classes,
            ),
        )

    def forward(self, x):
        x = x.view(
            x.size(0),
            -1,
        )

        x = self.fc(x)
        return x


In [ ]:
%%writefile /kaggle/working/cmr_har_code/models/resnet18.py
import torch
import torch.nn as nn


class ResNet18(nn.Module):
    def __init__(
        self,
        input_shape=None,
        num_classes=22,
        **kwargs,
    ):
        super(ResNet18, self).__init__()

        self.reshape = nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=3, kernel_size=(15, 23), stride=(3, 9), padding=(0, 0)),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels=3, out_channels=64, kernel_size=(3, 23), stride=1, padding=(0, 0)),
            nn.ReLU(inplace=True)
        )

        try:
            import torchvision.models as torchvision_models
        except ImportError as exc:
            raise ImportError(
                "torchvision is required for ResNet18. Install torchvision or remove resnet18 from the benchmark models."
            ) from exc

        resnet = torchvision_models.resnet18(weights=None)

        self.max_pool = resnet.maxpool
        self.layer1 = resnet.layer1
        self.layer2 = resnet.layer2
        self.layer3 = resnet.layer3
        self.layer4 = resnet.layer4

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512, num_classes)

    def forward(self, x):
        if x.dim() == 5:
            b, c1, c2, h, w = x.shape
            x = x.view(b, c1 * c2, h, w)

        x = self.reshape(x)
        x = self.max_pool(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)

        return x


In [ ]:
%%writefile /kaggle/working/cmr_har_code/models/resnet1d.py
import torch
import torch.nn as nn


def conv3x3(in_planes, out_planes, stride=1, group=1):
    """3x3 convolution with padding"""
    return nn.Conv1d(
        in_planes,
        out_planes,
        kernel_size=3,
        stride=stride,
        padding=1,
        bias=False,
        groups=group,
    )


def conv1x1(in_planes, out_planes, stride=1, group=1):
    """1x1 convolution"""
    return nn.Conv1d(
        in_planes,
        out_planes,
        kernel_size=1,
        stride=stride,
        bias=False,
        groups=group,
    )


class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, inplanes, planes, stride=1, group=1, downsample=None):
        super(BasicBlock, self).__init__()
        self.conv1 = conv3x3(inplanes, planes, stride, group=group)
        self.bn1 = nn.BatchNorm1d(planes)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = conv3x3(planes, planes, group=group)
        self.bn2 = nn.BatchNorm1d(planes)
        self.downsample = downsample
        self.stride = stride

    def forward(self, x):
        identity = x
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.conv2(out)
        out = self.bn2(out)

        if self.downsample is not None:
            identity = self.downsample(x)

        out += identity
        out = self.relu(out)
        return out


class Bottleneck(nn.Module):
    expansion = 4

    def __init__(self, inplanes, planes, stride=1, group=1, downsample=None):
        super(Bottleneck, self).__init__()
        self.conv1 = conv1x1(inplanes, planes, group=group)
        self.bn1 = nn.BatchNorm1d(planes)
        self.conv2 = conv3x3(planes, planes, stride, group=group)
        self.bn2 = nn.BatchNorm1d(planes)
        self.conv3 = conv1x1(planes, planes * self.expansion, group=group)
        self.bn3 = nn.BatchNorm1d(planes * self.expansion)
        self.relu = nn.ReLU(inplace=True)
        self.downsample = downsample
        self.stride = stride

    def forward(self, x):
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)
        out = self.relu(out)

        out = self.conv3(out)
        out = self.bn3(out)

        if self.downsample is not None:
            identity = self.downsample(x)

        out += identity
        out = self.relu(out)
        return out


class ResNet1D(nn.Module):
    def __init__(self, block, layers, inchannel=270, num_classes=55):
        super(ResNet1D, self).__init__()
        self.inplanes = 128
        self.conv1 = nn.Conv1d(
            inchannel,
            128,
            kernel_size=7,
            stride=2,
            padding=3,
            bias=False,
            groups=1,
        )
        self.bn1 = nn.BatchNorm1d(128)
        self.conv2 = nn.Conv1d(
            128,
            128,
            kernel_size=7,
            stride=2,
            padding=3,
            bias=False,
        )
        self.bn2 = nn.BatchNorm1d(128)
        self.conv3 = nn.Conv1d(
            128,
            128,
            kernel_size=7,
            stride=2,
            padding=3,
            bias=False,
        )
        self.bn3 = nn.BatchNorm1d(128)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool1d(kernel_size=3, stride=2, padding=1)

        self.layer1 = self._make_layer(block, 128, layers[0], stride=1, group=1)
        self.layer2 = self._make_layer(block, 128, layers[1], stride=2, group=1)
        self.layer3 = self._make_layer(block, 256, layers[2], stride=2, group=1)
        self.layer4 = self._make_layer(block, 512, layers[3], stride=2, group=1)
        self.conv4 = conv3x3(512, 512, stride=2)
        self.avg_pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(512 * block.expansion, num_classes)

    def _make_layer(self, block, planes, blocks, stride=1, group=1):
        downsample = None
        if stride != 1 or self.inplanes != planes * block.expansion:
            downsample = nn.Sequential(
                conv1x1(self.inplanes, planes * block.expansion, stride),
                nn.BatchNorm1d(planes * block.expansion),
            )

        layers = []
        layers.append(block(self.inplanes, planes, stride, group, downsample))
        self.inplanes = planes * block.expansion
        for _ in range(1, blocks):
            layers.append(block(self.inplanes, planes, group=group))

        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        output = self.avg_pool(x)
        output = output.view(output.size(0), -1)
        output = self.fc(output)
        return output


def resnet18_1d(inchannel=270, num_classes=55):
    return ResNet1D(BasicBlock, [2, 2, 2, 2], inchannel=inchannel, num_classes=num_classes)


In [ ]:
%%writefile /kaggle/working/cmr_har_code/models/vit.py
from .base import BaseModel


class ViT(BaseModel):
    def __init__(
        self,
        input_shape,
        num_classes,
        **kwargs,
    ):
        super().__init__(
            input_shape=input_shape,
            num_classes=num_classes,
        )

        raise NotImplementedError(
            "ViT has not been implemented yet."
        )

    def forward(self, x):
        raise NotImplementedError


In [ ]:
%%writefile /kaggle/working/cmr_har_code/models/wiformer.py
"""
CSI-WiFormer (v5) - see WiFormer_Architecture.md

Input (from datasets.py)
------------------------
2d : (B, N_rx * N_ant * N_sub, T)     -- default loader output
3d : (B, N_rx * N_ant, N_sub, T)
4d : (B, N_rx, N_ant, N_sub, T)

Pipeline
--------
select antenna[0]  -> (B, N_rx, N_sub, T)
per-sample z-score -> (B, N_rx, N_sub, T)
Block 1 : unfold(W, S) + [Conv1d branch || FFT branch] -> fuse -> (B, N_rx, n_win, N_sub, D)
Block 2 : joint (Rx x Sub) attention per window, PE_sub only, CLS -> (B, n_win, D)
Block 3 : CLS + ALiBi temporal Transformer -> (B, D)
Classifier : LayerNorm -> Dropout -> Linear -> logits
             (Softmax is applied inside CrossEntropyLoss / at inference)
"""

import math

import torch
import torch.nn as nn
import torch.nn.functional as F

from .base import BaseModel


# ============================================================
# Dataset configs: physical layout of the channel axis
# Channel order in datasets.py is (rx, antenna, sub) row-major.
# ============================================================
WIFORMER_DATASET_CFG = {
    # 3 RX x 1 antenna x 56 sub
    "sshar_esp": {"n_rx": 3, "n_ant": 1, "n_sub": 56},
    # 3 RX x 4 antenna x 56 sub -> keep antenna[0]
    "sshar_asus": {"n_rx": 3, "n_ant": 4, "n_sub": 56},
    # 3 RX x 3 antenna x 30 sub (270 channels) -> keep antenna[0]
    "xrf55": {"n_rx": 3, "n_ant": 3, "n_sub": 30},
    # 1 RX x 3 antenna x 30 sub: treat the 3 antennas as 3 views
    "ut_har": {"n_rx": 3, "n_ant": 1, "n_sub": 30},
}


class SafeTransformerEncoderLayer(nn.TransformerEncoderLayer):
    """
    nn.TransformerEncoderLayer that never takes PyTorch's eval-mode "fast path".

    With a per-head float mask (ALiBi, shape (B*H, L, L)) the fused fast path
    (torch._transformer_encoder_layer_fwd / _native_multi_head_attention)
    returns the same output for every sample in eval mode -> constant
    predictions at test time (acc = 1/num_classes). Same parameters /
    state_dict keys as the parent class.
    """

    def forward(self, src, src_mask=None, src_key_padding_mask=None, is_causal=False):
        x = src
        if self.norm_first:
            x = x + self._safe_sa(self.norm1(x), src_mask, src_key_padding_mask)
            x = x + self._ff_block(self.norm2(x))
        else:
            x = self.norm1(x + self._safe_sa(x, src_mask, src_key_padding_mask))
            x = self.norm2(x + self._ff_block(x))
        return x

    def _safe_sa(self, x, attn_mask, key_padding_mask):
        # Call the functional MHA directly so MultiheadAttention's own
        # fast path is bypassed as well.
        attn = self.self_attn
        q = x.transpose(0, 1)                              # (L, B, D)
        out, _ = F.multi_head_attention_forward(
            q, q, q,
            attn.embed_dim, attn.num_heads,
            attn.in_proj_weight, attn.in_proj_bias,
            attn.bias_k, attn.bias_v, attn.add_zero_attn,
            attn.dropout, attn.out_proj.weight, attn.out_proj.bias,
            training=self.training,
            key_padding_mask=key_padding_mask,
            need_weights=False,
            attn_mask=attn_mask,
        )
        return self.dropout1(out.transpose(0, 1))


def make_encoder_layer(d_model, n_heads, dim_ff, dropout, activation, norm_first):
    return SafeTransformerEncoderLayer(
        d_model=d_model,
        nhead=n_heads,
        dim_feedforward=dim_ff,
        dropout=dropout,
        activation=activation,
        batch_first=True,
        norm_first=norm_first,
    )


# ============================================================
# Block 1 - Sub feature extractor
# ============================================================
class SubFeatureExtractor(nn.Module):
    """
    Two parallel branches on every (rx, window, sub) segment of length W,
    fused back to D (see CSI-WiFormer_Block1_FFT.md):

    - temporal : shared Conv1d stack -> Linear(feat, D)          -> v_time
    - spectral : rfft -> log1p|X| -> drop DC -> Linear(W//2, D)   -> v_freq
    - fuse     : Linear(2D, D) on [v_time || v_freq]

    use_time / use_freq allow ablation (a disabled branch is not built;
    the fuse layer then maps D -> D). With both disabled, Block 1 is removed
    and replaced by a plain Linear(W, D) patch embedding of the raw window.

    Capacity knobs (temporal branch)
    --------------------------------
    n_conv : 2 (default) or 3 -- the 3rd Conv1d(feat, feat, 3) keeps length W/4
    pool   : "avg"  -> global average over the W/4 steps (default, order-free)
             "flatN" (e.g. "flat4") -> average to N steps, then flatten, so the
             order inside the window survives (rising vs falling motion)
    """

    def __init__(self, d_model=64, feat=64, window=64, stride=32,
                 use_time=True, use_freq=True, n_conv=2, pool="avg"):
        super().__init__()
        self.window = window
        self.stride = stride
        self.d_model = d_model
        self.use_time = use_time
        self.use_freq = use_freq

        # Branch 1: temporal (Conv1d stack)
        if use_time:
            if n_conv not in (2, 3):
                raise ValueError(f"k1_conv must be 2 or 3, got {n_conv}")
            if pool == "avg":
                n_pool = 1
            elif pool.startswith("flat") and pool[4:].isdigit():
                n_pool = int(pool[4:])
            else:
                raise ValueError(f"k1_pool must be avg or flatN, got '{pool}'")
            c1 = max(32, feat // 2)
            layers = [
                nn.Conv1d(1, c1, kernel_size=8, stride=2, padding=3),     # 64 -> 32
                nn.BatchNorm1d(c1),
                nn.GELU(),
                nn.Conv1d(c1, feat, kernel_size=4, stride=2, padding=1),  # 32 -> 16
                nn.BatchNorm1d(feat),
                nn.GELU(),
            ]
            if n_conv == 3:
                layers += [
                    nn.Conv1d(feat, feat, kernel_size=3, stride=1, padding=1),  # 16 -> 16
                    nn.BatchNorm1d(feat),
                    nn.GELU(),
                ]
            layers.append(nn.AdaptiveAvgPool1d(n_pool))                   # 16 -> n_pool
            self.encoder = nn.Sequential(*layers)
            self.proj_time = nn.Linear(feat * n_pool, d_model)

        # Branch 2: spectral (FFT). rfft(W) -> W//2+1 bins, drop DC -> W//2
        if use_freq:
            self.proj_freq = nn.Linear(window // 2, d_model)

        # Fuse [v_time || v_freq] -> D  (single branch: Linear(D, D))
        n_branch = int(use_time) + int(use_freq)
        if n_branch > 0:
            self.fuse = nn.Linear(n_branch * d_model, d_model)
        else:
            # Block 1 removed: raw window -> D
            self.proj_raw = nn.Linear(window, d_model)

    def forward(self, x):
        # x: (B, N_rx, N_sub, T)
        B, R, S, _ = x.shape
        x = x.unfold(dimension=-1, size=self.window, step=self.stride)
        # (B, R, S, n_win, W)
        n_win = x.shape[3]
        x = x.permute(0, 1, 3, 2, 4)                       # (B, R, n_win, S, W)
        x = x.reshape(B * R * n_win * S, self.window)      # (N, W), shared by both branches
        feats = []

        # --- temporal ---
        if self.use_time:
            v_time = self.encoder(x.unsqueeze(1)).flatten(1)   # (N, F * n_pool)
            feats.append(self.proj_time(v_time))               # (N, D)

        # --- spectral ---
        if self.use_freq:
            # FFT in float32 (rfft does not support half on all backends)
            x_f = torch.fft.rfft(x.float(), dim=-1)            # (N, W//2+1) complex
            x_f = torch.log1p(x_f.abs())[:, 1:].to(x.dtype)    # drop DC -> (N, W//2)
            feats.append(self.proj_freq(x_f))                  # (N, D)

        # --- fuse ---
        if feats:
            out = self.fuse(torch.cat(feats, dim=-1))          # (N, D)
        else:
            out = self.proj_raw(x)                             # (N, D)
        return out.reshape(B, R, n_win, S, -1)               # (B, R, n_win, S, D)


# ============================================================
# Block 2 - Joint (Rx x Sub) attention per window (PE_sub only)
# ============================================================
class JointRxSubAttention(nn.Module):
    """
    Ablation knobs
    --------------
    mode : "joint"  - attention over all R*S tokens of a window (default)
           "per_rx" - attention over the S tokens of each Rx, then mean of the R CLS
           "pool"   - no attention: mean over the R*S tokens (also when n_layers=0)
    pe   : "sub"    - PE_sub only (default)
           "rx_sub" - PE_sub + learnable PE_rx
           "none"   - no positional embedding
    """

    def __init__(
        self,
        n_sub,
        d_model=64,
        n_heads=4,
        n_layers=1,
        dim_ff=256,
        dropout=0.1,
        activation="gelu",
        norm_first=True,
        n_rx=None,
        mode="joint",
        pe="sub",
    ):
        super().__init__()
        assert d_model % 2 == 0
        if mode not in ("joint", "per_rx", "pool"):
            raise ValueError(f"k2_mode must be joint/per_rx/pool, got '{mode}'")
        if pe not in ("sub", "rx_sub", "none"):
            raise ValueError(f"k2_pe must be sub/rx_sub/none, got '{pe}'")
        if n_layers == 0:
            mode = "pool"
        self.n_sub = n_sub
        self.mode = mode
        self.pe = pe

        # PE_sub: Embedding(N_sub, D/2) -> Linear(D/2, D).
        if pe in ("sub", "rx_sub"):
            self.embed_sub = nn.Embedding(n_sub, d_model // 2)
            self.pe_proj = nn.Linear(d_model // 2, d_model)
        if pe == "rx_sub":
            if n_rx is None:
                raise ValueError("k2_pe=rx_sub needs n_rx")
            self.embed_rx = nn.Embedding(n_rx, d_model)

        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        if mode != "pool":
            self.attn_layer = nn.TransformerEncoder(
                make_encoder_layer(
                    d_model, n_heads, dim_ff, dropout, activation, norm_first
                ),
                num_layers=n_layers,
                enable_nested_tensor=False,
            )

    def _pos(self, R, S, device):
        # (R*S, D) or None
        if self.pe == "none":
            return None
        sub_idx = torch.arange(S, device=device)
        pe = self.pe_proj(self.embed_sub(sub_idx))          # (S, D)
        pe = pe.unsqueeze(0).expand(R, -1, -1)              # (R, S, D)
        if self.pe == "rx_sub":
            rx_idx = torch.arange(R, device=device)
            pe = pe + self.embed_rx(rx_idx).unsqueeze(1)
        return pe.reshape(R * S, -1)

    def forward(self, x):
        # x: (B, R, n_win, S, D)
        B, R, n_win, S, D = x.shape
        x = x.permute(0, 2, 1, 3, 4).reshape(B * n_win, R * S, D)

        pe = self._pos(R, S, x.device)
        if pe is not None:
            x = x + pe.unsqueeze(0)                         # broadcast B*n_win

        if self.mode == "pool":
            return x.mean(dim=1).reshape(B, n_win, D)

        if self.mode == "per_rx":
            x = x.reshape(B * n_win * R, S, D)
            cls = self.cls_token.expand(x.shape[0], -1, -1)
            x = self.attn_layer(torch.cat([cls, x], dim=1))
            return x[:, 0].reshape(B, n_win, R, D).mean(dim=2)

        cls = self.cls_token.expand(B * n_win, -1, -1)
        x = torch.cat([cls, x], dim=1)                      # (B*n_win, 1+R*S, D)
        x = self.attn_layer(x)
        return x[:, 0].reshape(B, n_win, D)                 # (B, n_win, D)


# ============================================================
# Block 3 - Temporal Transformer with ALiBi
# ============================================================
def get_alibi_slopes(n_heads):
    return 2 ** (-8 / n_heads * torch.arange(1, n_heads + 1, dtype=torch.float32))


class TemporalTransformer(nn.Module):
    """
    Ablation knobs
    --------------
    pos : "alibi"   - ALiBi bias (default)
          "learned" - learnable absolute PE added to the tokens (incl. CLS)
          "none"    - no positional information
    n_layers = 0    - no Transformer: mean over the n_win window tokens
    """

    def __init__(
        self,
        d_model=64,
        n_heads=4,
        n_layers=2,
        dim_ff=256,
        dropout=0.1,
        activation="gelu",
        norm_first=True,
        pos="alibi",
        max_len=None,
    ):
        super().__init__()
        if pos not in ("alibi", "learned", "none"):
            raise ValueError(f"k3_pos must be alibi/learned/none, got '{pos}'")
        self.n_heads = n_heads
        self.n_layers = n_layers
        self.pos = pos
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        if n_layers > 0:
            self.encoder = nn.TransformerEncoder(
                make_encoder_layer(
                    d_model, n_heads, dim_ff, dropout, activation, norm_first
                ),
                num_layers=n_layers,
                enable_nested_tensor=False,
            )
            if pos == "learned":
                if max_len is None:
                    raise ValueError("k3_pos=learned needs max_len")
                self.pos_embed = nn.Parameter(torch.zeros(1, max_len, d_model))
        self.register_buffer(
            "alibi_slopes", get_alibi_slopes(n_heads), persistent=False
        )

    def build_alibi_bias(self, seq_len, device, dtype):
        # alibi[h, i, j] = -m_h * |i - j| over all seq_len = n_win + 1 tokens
        pos = torch.arange(seq_len, device=device)
        dist = (pos.unsqueeze(0) - pos.unsqueeze(1)).abs().to(dtype)
        return -self.alibi_slopes.to(device, dtype).view(-1, 1, 1) * dist.unsqueeze(0)

    def forward(self, x):
        # x: (B, n_win, D)
        if self.n_layers == 0:
            return x.mean(dim=1)                            # (B, D)

        B = x.shape[0]
        cls = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1)                      # (B, 1+n_win, D)
        L = x.shape[1]

        mask = None
        if self.pos == "alibi":
            alibi = self.build_alibi_bias(L, x.device, x.dtype)  # (H, L, L)
            # nn.MultiheadAttention float mask: (B*H, L, L), batch-major
            mask = alibi.unsqueeze(0).expand(B, -1, -1, -1).reshape(B * self.n_heads, L, L)
        elif self.pos == "learned":
            x = x + self.pos_embed[:, :L]

        x = self.encoder(x, mask=mask)
        return x[:, 0]                                      # (B, D)


# ============================================================
# Full model
# ============================================================
class WiFormer(BaseModel):
    def __init__(
        self,
        input_shape,
        num_classes,
        dataset=None,
        n_rx=None,
        n_ant=None,
        n_sub=None,
        window=64,
        stride=32,
        d_model=64,
        feat=64,
        k1_conv=2,
        k1_pool="avg",
        k2_layers=2,
        k2_heads=4,
        k2_ff=None,
        k3_layers=4,
        k3_heads=4,
        k3_ff=None,
        dropout=0.1,
        activation="gelu",
        norm_first=True,
        sample_norm=True,
        k1_branch="both",
        k2_mode="joint",
        k2_pe="sub",
        k3_pos="alibi",
        ant_mode="first",
        **kwargs,
    ):
        super().__init__(input_shape=input_shape, num_classes=num_classes)

        cfg = dict(WIFORMER_DATASET_CFG.get((dataset or "").lower(), {}))
        n_rx = n_rx or cfg.get("n_rx")
        n_ant = n_ant or cfg.get("n_ant", 1)
        n_sub = n_sub or cfg.get("n_sub")
        if n_rx is None or n_sub is None:
            raise ValueError(
                f"WiFormer: unknown layout for dataset '{dataset}'. "
                "Pass n_rx / n_ant / n_sub explicitly."
            )
        self.n_rx, self.n_ant, self.n_sub = n_rx, n_ant, n_sub
        self.sample_norm = sample_norm
        if ant_mode not in ("first", "all"):
            raise ValueError(f"WiFormer: ant_mode must be first/all, got '{ant_mode}'")
        self.ant_mode = ant_mode
        # "all": every (rx, antenna) pair is its own view -> R' = R * A
        n_view = n_rx * n_ant if ant_mode == "all" else n_rx

        T = input_shape[-1]
        n_channels = math.prod(input_shape[:-1])
        if n_channels != n_rx * n_ant * n_sub:
            raise ValueError(
                f"WiFormer: input_shape {tuple(input_shape)} has {n_channels} "
                f"channels, expected n_rx*n_ant*n_sub = {n_rx}*{n_ant}*{n_sub}"
            )
        if T < window:
            raise ValueError(f"WiFormer: T={T} < window={window}")
        self.n_win = (T - window) // stride + 1

        # FFN width defaults to 4 * D (= 256 for D = 64, as before)
        k2_ff = k2_ff or 4 * d_model
        k3_ff = k3_ff or 4 * d_model
        for name, h in (("k2_heads", k2_heads), ("k3_heads", k3_heads)):
            if d_model % h:
                raise ValueError(f"WiFormer: d_model={d_model} not divisible by {name}={h}")

        if k1_branch not in ("both", "time", "freq", "none"):
            raise ValueError(f"WiFormer: k1_branch must be both/time/freq/none, got '{k1_branch}'")
        self.block1 = SubFeatureExtractor(
            d_model, feat, window, stride,
            use_time=k1_branch in ("both", "time"),
            use_freq=k1_branch in ("both", "freq"),
            n_conv=k1_conv, pool=k1_pool,
        )
        self.block2 = JointRxSubAttention(
            n_sub, d_model, k2_heads, k2_layers, k2_ff, dropout, activation, norm_first,
            n_rx=n_view, mode=k2_mode, pe=k2_pe,
        )
        self.block3 = TemporalTransformer(
            d_model, k3_heads, k3_layers, k3_ff, dropout, activation, norm_first,
            pos=k3_pos, max_len=self.n_win + 1,
        )
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Dropout(dropout),
            nn.Linear(d_model, num_classes),
        )

        self._init_weights()

    def _init_weights(self):
        # Linear: Xavier uniform · Conv: Kaiming · CLS / Embedding: trunc normal
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Embedding):
                nn.init.trunc_normal_(m.weight, std=0.02)
        # MHA packed in_proj (Q,K,V) is not an nn.Linear -> Xavier as well
        for m in self.modules():
            if isinstance(m, nn.MultiheadAttention):
                nn.init.xavier_uniform_(m.in_proj_weight)
                nn.init.zeros_(m.in_proj_bias)
        nn.init.trunc_normal_(self.block2.cls_token, std=0.02)
        nn.init.trunc_normal_(self.block3.cls_token, std=0.02)
        if hasattr(self.block3, "pos_embed"):
            nn.init.trunc_normal_(self.block3.pos_embed, std=0.02)

    def preprocess(self, x):
        # (B, C, T) / (B, R*A, S, T) / (B, R, A, S, T) -> (B, R, S, T)
        B, T = x.shape[0], x.shape[-1]
        x = x.reshape(B, self.n_rx, self.n_ant, self.n_sub, T)
        if self.ant_mode == "all":
            x = x.reshape(B, self.n_rx * self.n_ant, self.n_sub, T)
        else:
            x = x[:, :, 0]                                  # antenna[0] of each RX
        if self.sample_norm:
            # z-score per (rx, sub) per sample, along time
            mean = x.mean(dim=-1, keepdim=True)
            std = x.std(dim=-1, keepdim=True) + 1e-6
            x = (x - mean) / std
        return x

    def forward(self, x):
        x = self.preprocess(x)                              # (B, R, S, T)
        x = self.block1(x)                                  # (B, R, n_win, S, D)
        x = self.block2(x)                                  # (B, n_win, D)
        x = self.block3(x)                                  # (B, D)
        return self.classifier(x)                           # (B, n_class) logits


In [ ]:
import subprocess
# thop chỉ để đo FLOPs (tuỳ chọn, cần Internet); thiếu thì FLOPs = N/A
!pip install -q thop || echo 'thop not installed -> FLOPs = N/A'
!cd /kaggle/working/cmr_har_code && ls -R | head -30 && python ablation.py --list

## Chạy ablation

In [ ]:
import shutil, shlex, time

def run(cmd, cwd):
    """Chạy lệnh và in log ra notebook theo thời gian thực."""
    print('$', ' '.join(cmd), flush=True)
    proc = subprocess.Popen(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end='', flush=True)
    return proc.wait()

RESULT_BASE = f'{OUTPUT}/results/{RUN_TAG}'
cases = CASES if SPLIT_MODE in ('position', 'user') else [0]
t0 = time.time()
for case in cases:
    tag = SPLIT_MODE if SPLIT_MODE == 'repeat' else f'{SPLIT_MODE}_c{case}'
    result_root = f'{RESULT_BASE}/{tag}'
    os.makedirs(result_root, exist_ok=True)

    # chạy tiếp từ Output phiên trước: copy các run đã xong, --skip_done sẽ bỏ qua
    prev = Path(PREV_RESULTS) / tag if PREV_RESULTS else None
    if prev and prev.is_dir():
        shutil.copytree(prev, result_root, dirs_exist_ok=True)
        print('copied previous results from', prev)

    print(f'\n===== {DATASET} | {tag} | {VARIANTS} =====', flush=True)
    run(['python', '-u', 'ablation.py',
         f'--data_root={DATA_ROOT}', f'--dataset={DATASET}',
         f'--result_root={result_root}', '--skip_done',
         '--variants', *VARIANTS,
         '--extra', f'--split_mode={SPLIT_MODE}', f'--case_split={case}', *shlex.split(EXTRA)],
        cwd=CODE_DIR)
print(f'\nTotal time: {(time.time() - t0) / 3600:.2f} h')

## Tổng hợp tất cả run đã có (kể cả từ phiên trước)

In [ ]:
import pandas as pd
for csv in sorted(Path(RESULT_BASE).glob('*/ablation_*.csv')):
    print('\n#', csv.relative_to(RESULT_BASE))
    df = pd.read_csv(csv)
    cols = ['variant', 'accuracy', 'delta_acc', 'f1_macro', 'best_epoch', 'epochs_trained', 'total_params', 'training_time']
    df = df[[c for c in cols if c in df]]
    for c in ('accuracy', 'delta_acc', 'f1_macro'):
        if c in df: df[c] = (100 * df[c]).round(2)
    print(df.to_string(index=False))

## Đóng gói Output

In [ ]:
zip_path = shutil.make_archive(f'{OUTPUT}/results_{RUN_TAG}', 'zip', f'{OUTPUT}/results', RUN_TAG)
print(zip_path, f'{os.path.getsize(zip_path) / 1e6:.1f} MB')
shutil.rmtree(CODE_DIR, ignore_errors=True)
!ls -R {OUTPUT}/results | head -60